# PySpark

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

The five parts below read in order, and the last one points into the full
practice workbook merged at the bottom of this notebook.


## Part 1 - Orientation: what PySpark is and where it fits

No code in this part. Read these few cells first: they define the words every later
section uses - driver and executor, DataFrame and RDD, lazy plan - and give you a
30-second answer to "why PySpark at all?".


## What is PySpark and where does it fit in a data platform?

PySpark is the Python API for Apache Spark. It lets you write distributed data processing jobs in Python while Spark executes the work across a cluster.

In a modern lakehouse, PySpark is often used for:
+ Reading raw files from S3, ADLS, GCS, Kafka, or relational sources
+ Cleaning and standardizing semi-structured data
+ Joining large fact data with dimensions
+ Writing Delta, Iceberg, or Parquet tables for analytics
+ Running batch pipelines, streaming pipelines, and feature engineering jobs

Typical production flow:
```text
raw source files -> PySpark bronze job -> PySpark silver job -> Unity Catalog gold tables -> BI / ML / reverse ETL
```

Real-world example: an orders pipeline lands JSON in S3 every 15 minutes. A PySpark job reads the raw JSON, enforces a schema, deduplicates by `order_id`, joins customer data, and writes a governed table such as `main.gold.fact_order`.

## How do DataFrames differ from RDDs in PySpark?

DataFrames are structured, schema-aware distributed datasets. Spark can optimize DataFrame operations using Catalyst and Tungsten. RDDs are lower-level distributed collections where Spark sees less structure and has fewer optimization opportunities.

Interview answer: DataFrames are usually better for production pipelines because they support SQL optimization, column pruning, predicate pushdown, better execution planning, and integration with table formats.

How it works under the hood. Catalyst sees DataFrame expressions and rewrites them with rules such as predicate pushdown, column pruning and constant folding; Tungsten then compiles the physical operators into one generated Java method over a compact binary row format. An RDD holds opaque Python or JVM objects, so the optimizer cannot see inside a lambda, and `df.rdd` pays a per-row conversion between JVM rows and Python objects.

Worked example. A `filter` on a DataFrame compiles into a single generated method over binary rows. The same predicate written as `df.rdd.filter(lambda r: ...)` ships Python objects across the Py4J boundary for every row and can be 5-20x slower on a wide table.

Common mistake: `df.rdd.map(...).toDF()` to express logic that `pyspark.sql.functions` already covers. You keep the syntax you know but lose predicate pushdown and column pruning.

## When you would NOT use PySpark

Spark is a heavy default, not a habit. Reach for something simpler when:

+ the data fits in one machine's memory (a few GB) - pandas, DuckDB or plain SQL is faster to write, run and iterate on;
+ the work is pure relational SQL over warehouse tables - the warehouse engine (or dbt) avoids shipping and maintaining Python for what SQL already does;
+ latency-sensitive serving and row lookups are the job - pipelines are batch animals; point lookups belong in a database;
+ the source system can do the transform for you (a streaming engine, a view), because every extra engine multiplies the operational surface.

The honest two-line interview answer: "PySpark earns its cost once data volume, streaming,
or heavy custom Python logic appears; before that, simpler engines win."


## Part 2 - First session and the execution model

Everything above was words; here the machine appears. One idea to carry through this
part: **nothing runs until it must**. Create the session, read with an explicit schema,
transform lazily, and only then pull the trigger with an action.


## How do you create a SparkSession in PySpark?

`SparkSession` is the main entry point for DataFrame, SQL, catalog, and streaming operations. In Databricks notebooks, `spark` is usually already available. In jobs, tests, or local development, you create it explicitly.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("orders-pipeline")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In Databricks with Unity Catalog, you normally refer to tables with a three-part name:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders")

Interview point: `SparkSession` replaced older entry points like `SQLContext` for most DataFrame and SQL work. `SparkContext` still exists underneath, but most production PySpark code should use `SparkSession` and DataFrames.

How it works under the hood. `SparkSession.builder` reads static configs, then `getOrCreate()` returns the existing session if one is already running in this JVM. Configs passed to the builder are fixed for the life of the session; `spark.conf.set(...)` can change runtime SQL settings later. Every shuffle uses `spark.sql.shuffle.partitions` as its reduce fan-out.

Worked example, the defaults you should know:

```text
spark.sql.shuffle.partitions          200
spark.sql.autoBroadcastJoinThreshold  10485760 bytes (10 MB)
spark.sql.adaptive.enabled            true (Spark 3.2+)
spark.sql.files.maxPartitionBytes     134217728 bytes (128 MB)
```

Common mistake: calling `getOrCreate()` again with different configs and expecting them to apply. It silently returns the first session, so stop the old one or use `spark.conf.set` for runtime settings.

How it works under the hood. A PySpark program has one driver process that owns the `SparkSession`, and one or more executor processes that own the data. The driver turns your DataFrame calls into a logical plan, Catalyst rewrites it, and the DAG scheduler cuts it into stages at every shuffle. Executors run tasks on partitions and report metrics back; `master("local[*]")` packs driver and one executor into a single JVM, which is why it is a development tool and not a deployment topology.

Worked example. A nightly job reads 2 TB of Parquet in 200 files. The driver holds the plan and task metadata (a few MB), while executors hold shuffle files, cached blocks and broadcast tables. With a 2 GB driver and 16 GB executors, the driver limit is the plan plus anything you `collect()`; the executor limit is shuffle, cache and broadcast.

Common mistake: assuming PySpark runs Python on every worker. Python only runs for UDFs and driver-side code; the DataFrame engine runs in the JVM, which is exactly why built-ins are fast and Python UDFs are not.

Prefer DataFrames for most data engineering work:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

paid_orders = (
    spark.table("main.silver.orders")
    .where(F.col("status") == "paid")
    .select("order_id", "customer_id", "amount", "order_ts")
)

Use RDDs only when you truly need record-level custom distributed logic that DataFrame APIs cannot express:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
line_lengths = spark.sparkContext.textFile("s3://company-raw/logs/").map(len)

A habit that doubles learning speed: before you run any code cell, predict what it
will print. The cells in this notebook show patterns against real platform tables;
the merged practice workbook at the end runs the same ideas live, so predict first
there, then check yourself.


## How do you read data with explicit schemas in PySpark?

Explicit schemas make pipelines safer and faster. If you let Spark infer schemas from files, the schema may change based on a small sample, a malformed record, or a new partition.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import types as T

order_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders_raw = (
    spark.read
    .schema(order_schema)
    .option("mode", "PERMISSIVE")
    .json("s3://company-raw/orders/ds=2026-10-04/")
)

For CSV, be explicit about headers, delimiters, quotes, and bad-record handling:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
customers = (
    spark.read
    .schema("customer_id string, email string, created_at timestamp")
    .option("header", "true")
    .option("escape", '"')
    .csv("s3://company-raw/customers/")
)

Real-world example: if `amount` is accidentally delivered as `"N/A"` in one file, schema inference can degrade the column to string. With an explicit decimal schema, invalid rows become visible as quality failures instead of silently changing the table contract.

How it works under the hood. Reading JSON without a schema launches an extra job that samples files to infer types, so the data is touched twice. An explicit `StructType` skips that pass and lets the reader decode straight into the declared types. `mode` controls corrupt records: `PERMISSIVE` keeps the raw text in a corrupt-record column, `DROPMALFORMED` drops rows silently, and `FAILFAST` raises.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders_with_corrupt = (
    spark.read
    .schema(order_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json("s3://company-raw/orders/ds=2026-10-04/")
)

good_orders = orders_with_corrupt.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
bad_orders = orders_with_corrupt.filter(F.col("_corrupt_record").isNotNull())
print("good:", good_orders.count(), "corrupt:", bad_orders.count())

Worked example. Schema inference on a 2 TB JSON dataset costs one extra scan before the real job runs. If the last of three files delivers `amount` as `"N/A"`, inference over a sample can settle on `string`, while the explicit `decimal(12,2)` schema keeps the contract and turns the bad row into a countable null or corrupt record.

Common mistake: a mismatched type under `PERMISSIVE` produces nulls rather than an error, so always count the corrupt-record column (or nulls) after a load.

## What are transformations and actions in PySpark?

Transformations build a logical plan. Actions trigger execution.

Transformations:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

daily_revenue = (
    spark.table("main.silver.orders")
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)

No cluster job has to run yet. An action triggers execution:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
daily_revenue.count()
daily_revenue.write.mode("overwrite").saveAsTable("main.gold.daily_revenue")

Common transformations include `select`, `where`, `withColumn`, `join`, `groupBy`, `agg`, `distinct`, and `dropDuplicates`. Common actions include `count`, `collect`, `show`, `take`, `write`, and `foreachBatch`.

How it works under the hood. Transformations only add nodes to the logical plan; nothing is scheduled. An action triggers Catalyst optimization, physical planning, a DAG split into stages at each shuffle, and one task per partition. That is why the same DataFrame walked twice by two actions is computed twice unless you cached it.

Worked example. A `groupBy("order_date").agg(sum)` over 500 GB shows `HashAggregate(partial) -> Exchange -> HashAggregate(final)`. The partial aggregate runs inside each map task and shrinks the rows before the network, so the shuffle moves far less than 500 GB; that two-node pattern is the signature of a decomposable aggregate.

Common mistake: calling `count()` or `show()` between every step to "check progress". Each call re-runs the whole lineage from the source; cache the frame or read `explain("formatted")` instead.

## How does lazy evaluation work in PySpark?

Lazy evaluation means Spark waits until an action before executing the plan. This lets Spark optimize the whole plan instead of executing every line immediately.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders")

result = (
    orders
    .where("order_ts >= timestamp '2026-10-01'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)

result.explain("formatted")

Spark can combine filters, prune unused columns, push predicates to Parquet/Delta/Iceberg readers, and choose join strategies before running tasks.

Real-world example: if you read a 2 TB orders table but select only `order_id` and `amount`, Spark can avoid reading unused columns from Parquet files. That optimization is possible because the plan is declarative.

How it works under the hood. `explain("formatted")` prints the analyzed and optimized physical plan without reading data. Catalyst applies rule-based passes (predicate pushdown, constant folding, column pruning, null propagation) and the planner chooses join and aggregate strategies from table statistics. With AQE on, the printed plan is provisional and can change at runtime.

Worked example. `.where(ts >= ...).select(a, b).where(a > 0)` collapses into one `FileScan parquet` node with `PushedFilters: [IsNotNull(a), GreaterThan(a,0)]` and `ReadSchema: a,b`. Two filters and a projection became a single scan; no extra stage was created.

Common mistake: trusting the initial plan as final. AQE plans print `isFinalPlan=false` until an action runs, and `explain("cost")` shows the row estimates the planner used.

## How do you use Spark SQL and temp views from PySpark?

PySpark and Spark SQL work together. DataFrames can be registered as temporary views, and SQL results come back as DataFrames.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders")
orders.createOrReplaceTempView("orders")

daily = spark.sql("""
select
  date(order_ts) as order_date,
  count(*) as order_count,
  sum(amount) as revenue
from orders
where status in ('paid', 'shipped')
group by date(order_ts)
""")

For governed production tables, use Unity Catalog names directly:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create or replace table main.gold.daily_revenue as
select
  date(order_ts) as order_date,
  sum(amount) as revenue
from main.silver.orders
where status in ('paid', 'shipped')
group by date(order_ts)
""")

Interview point: PySpark DataFrame code and Spark SQL usually compile into the same execution engine. Choose the style that is clearer for the team and workload.

How it works under the hood. `createOrReplaceTempView` registers a name in the session catalog that points at the DataFrame's logical plan. `spark.sql(...)` parses SQL into the same kind of plan tree, so both paths meet before Catalyst optimization; a DataFrame and its SQL twin usually produce identical physical plans. A global temp view lives under `global_temp` and is visible across sessions in the same application.

Worked example. Register `orders` and run the DataFrame aggregate and the SQL aggregate side by side; both `explain()` outputs show the same `HashAggregate + Exchange` shape because they compile to the same plan.

Common mistake: temp views are session-scoped and disappear with the session, so a job that re-registers a stale plan under the same name can silently query old logic. Drop the view or use a fresh name when the underlying frame changes.

## How do you clean and standardize data in PySpark?

Use built-in column functions instead of Python row loops. Built-ins run inside Spark's optimized execution engine.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

clean_customers = (
    spark.table("main.bronze.customers")
    .select(
        F.trim(F.lower("email")).alias("email"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.to_timestamp("created_at").alias("created_at"),
        F.regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    )
    .where(F.col("email").contains("@"))
)

For reusable business logic, wrap DataFrame transformations in plain Python functions:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
def standardize_orders(df):
    return (
        df
        .withColumn("status", F.lower(F.trim("status")))
        .withColumn("order_date", F.to_date("order_ts"))
        .where(F.col("order_id").isNotNull())
    )

orders = standardize_orders(spark.table("main.bronze.orders"))

This style is testable, readable, and avoids hiding Spark logic inside side-effect-heavy notebooks.

How it works under the hood. Built-in string and timestamp functions are Catalyst expressions evaluated on Tungsten rows inside the JVM; `regexp_replace` becomes a Java regex call per row with no Python involved. Wrapping transformations in plain Python functions is fine because those functions run once on the driver to build the plan, not once per row.

Worked example:

```text
"  Ada@Example.COM "  -> lower(trim(...))        -> "ada@example.com"
"+1 (312) 555-0100"   -> regexp_replace([^0-9])   -> "13125550100"
```

Common mistake: reaching for `F.udf` to trim, lowercase or split. The built-in version is shorter, faster, and keeps whole-stage code generation intact.

## Part 3 - Shape the data

You can now predict what will and will not execute. Part 3 is where frames meet data:
joins, aggregations and window functions are the wide operations that move bytes across
the cluster, and the cookbook closes it as the everyday recipe list.


## How do you join data in PySpark?

A join combines DataFrames by key. The most important production questions are key uniqueness, null handling, data skew, and whether Spark should shuffle both sides or broadcast the small side.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.silver.orders")
customers = spark.table("main.silver.customers")

enriched = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="left")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.customer_segment",
        "o.order_ts",
        "o.amount",
    )
)

If the dimension is small enough, broadcast it:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql.functions import broadcast

enriched = orders.join(broadcast(customers), "customer_id", "left")

Real-world example: broadcasting a 20 MB customer segment dimension can avoid a large shuffle when joining it to a 3 TB orders fact table. Do not broadcast a dimension just because it is called a dimension; verify its size.

How it works under the hood. A sort-merge join shuffles both sides by the key hash, sorts each partition, then merges. A broadcast hash join ships the small side to every executor, builds a hash table there, and streams the large side past it with no shuffle at all. Spark picks broadcast when the estimated side size is under `spark.sql.autoBroadcastJoinThreshold`, and a `broadcast()` hint overrides the estimate.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

orders = spark.table("main.silver.orders").select(
    "order_id", "customer_id", "order_ts", "amount", "status"
)
customers = spark.table("main.silver.customers").select(
    "customer_id", "segment", "country_code"
)

enriched = (
    orders.alias("o")
    .join(broadcast(customers).alias("c"), on="customer_id", how="left")
    .select("o.order_id", "o.amount", "o.status", "c.segment", "c.country_code")
)
enriched.explain("formatted")  # expect BroadcastHashJoin, not SortMergeJoin

Worked example. A 3 TB orders fact joined to a 20 MB customer dimension: sort-merge shuffles about 3 TB, while broadcast sends 20 MB to each executor and shuffles nothing. Across 100 executors that is about 2 GB of network instead of terabytes.

Common mistake: broadcasting a "dimension" that is actually 2 GB. Every executor must hold the whole build side, so you get OOM or a silently ignored hint; confirm `BroadcastHashJoin` in the plan.

## How do you aggregate data in PySpark?

Aggregations summarize records by one or more grouping columns. They usually create a shuffle because rows with the same key must be brought together.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

daily_product_sales = (
    spark.table("main.gold.fact_order_line")
    .groupBy("order_date", "product_id")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("quantity").alias("units"),
        F.sum("net_amount").alias("revenue"),
    )
)

For conditional metrics, use expressions:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
customer_metrics = (
    spark.table("main.gold.fact_order")
    .groupBy("customer_id")
    .agg(
        F.sum("amount").alias("lifetime_value"),
        F.sum(F.when(F.col("status") == "refunded", F.col("amount")).otherwise(0)).alias("refunded_amount"),
        F.max("order_ts").alias("last_order_ts"),
    )
)

Interview point: aggregations often need tuning because they shuffle data. Check the Spark UI for skewed shuffle partitions, spilled memory, and long-running reduce tasks.

How it works under the hood. A hash aggregate first runs a partial aggregate inside each map task, exchanges the reduced rows by group key, then runs the final aggregate. That partial step is why `sum` and `count` shuffle so much less than `countDistinct`, which must ship the distinct values themselves because they cannot be combined from partial sets exactly.

Worked example. 1 billion rows at 100 bytes each is 100 GB; a partial aggregate over 50 million keys can cut that to roughly 200 million intermediate rows, so the shuffle carries about 20 GB instead of 100 GB.

Common mistake: `countDistinct` on a high-cardinality column inside a big `groupBy`. Use `approx_count_distinct` when a small error is acceptable, or deduplicate first and count rows.

## How do window functions work in PySpark?

Window functions calculate values across related rows without collapsing the result like `groupBy` does.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import Window, functions as F

w = Window.partitionBy("customer_id").orderBy(F.col("order_ts").desc())

latest_order = (
    spark.table("main.silver.orders")
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") == 1)
    .drop("rn")
)

Common window use cases:

+ Deduplicating CDC records by latest sequence number
+ Finding the latest row per business key
+ Calculating rolling metrics
+ Ranking products or customers within a segment

CDC example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
w = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(),
    F.col("ingested_at").desc(),
)

deduped_cdc = (
    spark.table("main.bronze.orders_cdc")
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

How it works under the hood. A window with `partitionBy` plans as `Exchange hashpartitioning(key)` followed by `Sort` inside each partition, then the window expression runs in one pass. Window expressions without `partitionBy` send every row to a single partition, which turns a distributed job into one task. `row_number` needs the sort; `sum().over(rowsBetween(...))` reuses it.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import Window, functions as F

seven_days = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("order_ts").cast("long"))
    .rangeBetween(-7 * 86400, 0)
)

rolling = (
    spark.table("main.silver.orders")
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("spend_7d", F.sum("amount").over(seven_days))
)

Worked example. A 500 GB events table deduplicated with `row_number` over `partitionBy(order_id).orderBy(ts desc)` shuffles the whole table once, even if most keys are unique and there is nothing to deduplicate.

Common mistake: omitting a deterministic tie-break in `orderBy`, so the "latest" row changes between runs. Add the source sequence number or another column that breaks ties the same way every time.

## What are common PySpark data manipulation examples?

Most PySpark pipeline work is ordinary data manipulation: parsing timestamps, deriving dates, handling nulls, standardizing strings, splitting arrays, and creating business flags. Prefer `pyspark.sql.functions` because those expressions stay inside Spark's optimizer.

Start with a realistic orders DataFrame:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders = spark.table("main.bronze.orders")

**Dates and timestamps.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders_with_dates = (
    orders
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("order_month", F.date_trunc("month", "order_ts"))
    .withColumn("order_year", F.year("order_ts"))
    .withColumn("order_week", F.weekofyear("order_ts"))
    .withColumn("ship_by_date", F.date_add("order_date", 2))
    .withColumn("days_since_order", F.datediff(F.current_date(), "order_date"))
)

**Time zone conversion.** Common when sources emit UTC but reporting is local:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
localized = (
    orders
    .withColumn("order_ts_utc", F.to_timestamp("order_ts"))
    .withColumn(
        "order_ts_chicago",
        F.from_utc_timestamp("order_ts_utc", "America/Chicago"),
    )
    .withColumn("order_date_chicago", F.to_date("order_ts_chicago"))
)

**String cleanup.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
clean_customers = (
    spark.table("main.bronze.customers")
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("phone_digits", F.regexp_replace("phone", r"[^0-9]", ""))
    .withColumn("email_domain", F.element_at(F.split("email", "@"), -1))
)

**Null handling and default values.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders_filled = (
    orders
    .withColumn("discount_amount", F.coalesce("discount_amount", F.lit(0.0)))
    .withColumn("status", F.coalesce("status", F.lit("unknown")))
    .fillna({"country_code": "US", "currency": "USD"})
)

**Type casting and safe derived columns.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
typed_orders = (
    orders
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn(
        "unit_price",
        F.when(F.col("quantity") > 0, F.col("amount") / F.col("quantity")),
    )
)

**Conditional business flags.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
classified = (
    orders
    .withColumn(
        "order_size",
        F.when(F.col("amount") >= 1000, "enterprise")
         .when(F.col("amount") >= 100, "mid_market")
         .otherwise("small"),
    )
    .withColumn(
        "is_completed",
        F.col("status").isin("paid", "shipped", "delivered"),
    )
)

**Arrays and nested fields.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
events = spark.table("main.bronze.events")

items = (
    events
    .select(
        "event_id",
        F.col("user.id").alias("user_id"),
        F.explode_outer("items").alias("item"),
    )
    .select(
        "event_id",
        "user_id",
        F.col("item.sku").alias("sku"),
        F.col("item.quantity").alias("quantity"),
    )
)

**Parsing JSON strings.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
payload_schema = "source string, campaign struct<id:string,name:string>"

parsed = (
    events
    .withColumn("payload", F.from_json("payload_json", payload_schema))
    .select(
        "event_id",
        F.col("payload.source").alias("source"),
        F.col("payload.campaign.id").alias("campaign_id"),
    )
)

**Deduplicating records.**

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
deduped_orders = (
    orders
    .dropDuplicates(["order_id"])
)

**Latest record via a window (CDC).** For CDC or late-arriving updates, keep the
newest record by source ordering:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import Window

w = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(),
    F.col("ingested_at").desc(),
)

latest_orders = (
    orders
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

Real-world guidance: if the transformation is expressible with built-in functions, use built-in functions. They are easier for Spark to optimize than Python loops or UDFs.

How it works under the hood. Every `withColumn` adds a projection node, and each `dropDuplicates` or window adds a wide node that exchanges rows by key. Catalyst later fuses the narrow projections into one, but a dedupe still moves the whole dataset across the network. `dropDuplicates(["order_id"])` repartitions by `order_id`; the no-argument form compares entire rows.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

order_summary = (
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("customer_id", "order_date")
    .agg(
        F.count("*").alias("orders"),
        F.sum("amount").alias("revenue"),
        F.max("amount").alias("largest_order"),
    )
    .withColumn("revenue_per_order", F.round(F.col("revenue") / F.col("orders"), 2))
)

Worked example. Deduplicating a 1 TB orders table with `dropDuplicates(["order_id"])` shuffles roughly 1 TB by that key and keeps one arbitrary row per key.

Common mistake: expecting `dropDuplicates` to keep the newest record. It has no ordering, so the survivor is non-deterministic; for CDC use `row_number()` over a window ordered by a source sequence and a tie-break.

## How do you handle nested JSON and arrays in PySpark?

Nested data is common in event, clickstream, API, and CDC payloads. Use struct and array functions instead of converting rows to Python dictionaries.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

events = spark.read.json("s3://company-raw/events/")

flattened = events.select(
    F.col("event_id"),
    F.col("user.id").alias("user_id"),
    F.col("device.os").alias("device_os"),
    F.explode_outer("items").alias("item"),
)

items = flattened.select(
    "event_id",
    "user_id",
    F.col("item.sku").alias("sku"),
    F.col("item.quantity").alias("quantity"),
)

If a payload column contains JSON strings, parse it with `from_json`:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
payload_schema = "event_type string, campaign struct<id:string,name:string>"

parsed = (
    spark.table("main.bronze.raw_events")
    .withColumn("payload", F.from_json("payload_json", payload_schema))
    .select("event_id", "payload.event_type", "payload.campaign.id")
)

Interview point: `explode` can multiply rows. Always know the expected array size and watch for unexpected row explosions.

How it works under the hood. Parquet and JSON readers expose nested objects as `struct` and `array` columns, and `F.col("user.id")` resolves the field at plan time with no data conversion. `explode` is a `Generate` operator that expands one row into many inside its partition; it is narrow, but it multiplies the row count, so every later shuffle and write processes more data.

Worked example. An events file with an average array length of 4 and 100 million events becomes 400 million rows after `explode`, so a downstream join or write handles four times the rows and roughly four times the bytes.

Common mistake: choosing `explode` when `explode_outer` is correct. Rows with a null or empty array simply disappear, which is a classic silent correctness bug in event pipelines.

## Part 4 - Production patterns

The recipes above assume someone handed you clean tables. A production job asks who
wrote them and how: governed writes, incremental loads, tests, bad records, streaming,
monitoring, and the tuning levers that keep a nightly job boring.


## How do you write PySpark tables in Databricks Unity Catalog?

With Unity Catalog, prefer managed table names like `catalog.schema.table` over raw paths in application code. This keeps governance, lineage, discovery, and permissions at the table layer.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("create schema if not exists main.gold")

(
    daily_product_sales
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.gold.daily_product_sales")
)

For partitioned Delta tables:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    daily_product_sales
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("order_date")
    .saveAsTable("main.gold.daily_product_sales")
)

For table-format-aware writes, use `writeTo` when the target table already exists:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    daily_product_sales
    .writeTo("main.gold.daily_product_sales")
    .append()
)

Real-world guidance: do not scatter hard-coded S3 paths throughout jobs. Use table names in PySpark transformations, and reserve physical storage paths for ingestion boundaries, external locations, or infrastructure configuration.

How it works under the hood. `saveAsTable` goes through the catalog, which owns the schema, permissions and lineage. `writeTo(...).append()` works only when the table already exists and uses the table's stored schema and partitioning. On a partitioned table, `mode("overwrite")` replaces the whole table unless `spark.sql.sources.partitionOverwriteMode` is `dynamic`, in which case only partitions present in the DataFrame are replaced.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    daily_product_sales
    .repartition("order_date")
    .sortWithinPartitions(F.col("product_id"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_date")
    .saveAsTable("main.gold.daily_product_sales")
)

Worked example. A table partitioned by `order_date` with 365 days and one new day of data: a static overwrite rewrites all 365 partitions, a dynamic overwrite touches only the one partition present in the frame.

Common mistake: leaving `overwriteSchema` on while an upstream type drifts. The write succeeds and quietly changes the table contract; own the schema in the pipeline and fail fast instead.

## How do you build an incremental PySpark pipeline?

An incremental pipeline processes only new or changed records. The usual pattern is to use a watermark, ingestion timestamp, source sequence number, or table change feed.

Simple batch watermark example:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

last_loaded_ts = (
    spark.table("main.ops.pipeline_watermarks")
    .where(F.col("pipeline_name") == "orders_silver")
    .select(F.max("loaded_until_ts").alias("ts"))
    .first()["ts"]
)

new_orders = (
    spark.table("main.bronze.orders")
    .where(F.col("ingested_at") > F.lit(last_loaded_ts))
)

Merge into the target table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
new_orders.createOrReplaceTempView("new_orders")

spark.sql("""
merge into main.silver.orders as target
using new_orders as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert *
""")

Interview point: a reliable incremental job needs idempotency. If the job retries the same batch, the target should end in the same state, not duplicate data.

How it works under the hood. The watermark read (`max(loaded_until_ts)`) and the `MERGE` are separate actions: the first is a small job that returns one row to the driver, the second is a full job. Delta and Iceberg implement `MERGE` by rewriting the files that contain matched keys and committing a new table version, so idempotency comes from matching on the business key, not from row-by-row updates.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import Window, functions as F

staging_window = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(), F.col("ingested_at").desc()
)

staged = (
    new_orders
    .withColumn("rn", F.row_number().over(staging_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
staged.createOrReplaceTempView("staged_orders")

Worked example. A batch of 5 million changed orders merging into a 2 billion row table: with partition pruning on `order_date`, `MERGE` rewrites only the files in the touched partitions rather than the whole table. The job takes minutes instead of hours.

Common mistake: deriving the watermark from `max(ingested_at)` on the target itself. Late data that arrives after that value is skipped forever; keep the watermark in an ops table and use a deliberate lookback window.

## How do you test PySpark transformations?

Keep business transformations as functions that accept and return DataFrames. Then create small test DataFrames and compare outputs.

Transformation:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

def add_order_date(df):
    return df.withColumn("order_date", F.to_date("order_ts"))

Test example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
from datetime import datetime

input_df = spark.createDataFrame(
    [("o1", datetime(2026, 10, 4, 12, 30, 0))],
    ["order_id", "order_ts"],
)

actual = add_order_date(input_df).select("order_id", "order_date").collect()

assert str(actual[0]["order_date"]) == "2026-10-04"

For real projects, use `pytest`, fixtures for the Spark session, and small deterministic inputs. Test data should cover nulls, duplicate keys, late-arriving data, invalid records, and boundary timestamps.

How it works under the hood. A DataFrame test needs a session, but `createDataFrame` from local rows builds a small deterministic frame with no cluster involved. The test should compare collected rows in a defined order and also assert on schema and nullability, because a transformation that changes types is as much a bug as one that changes values.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
def test_add_order_date_keeps_nulls():
    input_df = spark.createDataFrame(
        [("o1", "2026-10-04 12:30:00"), ("o2", None)],
        "order_id string, order_ts string",
    )
    rows = add_order_date(input_df).orderBy("order_id").collect()
    assert [r["order_id"] for r in rows] == ["o1", "o2"]
    assert str(rows[0]["order_date"]) == "2026-10-04"
    assert rows[1]["order_date"] is None

Worked example. The one-row test above builds a one-partition local frame, runs a single action and finishes in milliseconds, so a suite of them fits in a normal CI run against one shared local `SparkSession` fixture.

Common mistake: asserting on `collect()` order without an `orderBy`. Spark does not guarantee row order, so the test passes locally and fails in CI for no real reason.

## How do you handle bad records and data quality checks in PySpark?

Bad records should be isolated, measurable, and visible. Do not silently drop them unless the product owner agrees that loss is acceptable.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders = spark.table("main.bronze.orders")

valid_orders = orders.where(
    F.col("order_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & (F.col("amount") >= 0)
)

invalid_orders = orders.subtract(valid_orders).withColumn(
    "rejected_at", F.current_timestamp()
)

invalid_orders.write.mode("append").saveAsTable("main.ops.rejected_orders")
valid_orders.write.mode("append").saveAsTable("main.silver.orders")

A more explicit rule-based pattern:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
checked = (
    orders
    .withColumn(
        "quality_error",
        F.when(F.col("order_id").isNull(), "missing_order_id")
         .when(F.col("amount") < 0, "negative_amount")
    )
)

checked.where("quality_error is null").write.mode("append").saveAsTable("main.silver.orders")
checked.where("quality_error is not null").write.mode("append").saveAsTable("main.ops.orders_quarantine")

Interview point: quality checks should happen before publishing gold tables. A dashboard should not discover broken data before the pipeline does.

How it works under the hood. A rule column evaluated once gives one plan with two filters over the same scan, so both the valid and the quarantine write read the source. `subtract` (and `exceptAll`) is a wide operation that compares whole rows and shuffles both sides, so it is heavier than an explicit `is_valid` flag and behaves differently once you add a timestamp column.

Worked example. 10 million raw rows with 200 thousand bad ones: caching the checked frame and writing valid plus quarantine reads the source once at the cost of holding the frame; recomputing reads twice but needs no cache memory.

Common mistake: using `subtract` after adding a `rejected_at` timestamp. The columns no longer match, so no row cancels out and the quarantine gets everything. Derive a boolean and filter it twice.

## How do you use UDFs safely in PySpark?

Use Spark built-in functions first. Python UDFs often move data between the JVM and Python worker, which can be slower and harder for Spark to optimize.

Prefer this:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

cleaned = customers.withColumn(
    "email_domain",
    F.element_at(F.split(F.lower("email"), "@"), -1),
)

Instead of this:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F, types as T

@F.udf(T.StringType())
def email_domain(email):
    if email is None or "@" not in email:
        return None
    return email.lower().split("@")[-1]

cleaned = customers.withColumn("email_domain", email_domain("email"))

If you need Python logic for vectorized numeric or ML work, consider Pandas UDFs, but measure performance and memory behavior.

In [ ]:
# ILLUSTRATIVE - requires pandas, pyspark: install to run, or read on.
import pandas as pd
from pyspark.sql import functions as F, types as T

@F.pandas_udf(T.DoubleType())
def cap_score(score: pd.Series) -> pd.Series:
    return score.clip(lower=0.0, upper=1.0)

scored = users.withColumn("capped_score", cap_score("score"))

How it works under the hood. A Python UDF plans as a `BatchEvalPython` node: each task starts a Python worker, sends batches of rows over a socket, and receives results, usually via Arrow or pickles. Catalyst cannot push predicates through that boundary and whole-stage code generation stops there, so the UDF is a black box in the middle of an otherwise compiled pipeline.

Worked example. For 100 million rows, a row-at-a-time UDF costing one microsecond of Python per row adds about 100 seconds of interpreter time before you count serialization, and it blocks the generated code path around it.

Common mistake: an `@F.udf` closure that captures a large lookup dict or DataFrame. The object is pickled to every task; broadcast it as a Spark broadcast variable instead, or express the logic with built-ins.

## How do you process streaming data with PySpark Structured Streaming?

Structured Streaming lets you express streaming logic with DataFrame APIs. Spark runs the query continuously or in micro-batches.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema("event_id string, user_id string, event_ts timestamp, event_type string")
    .load("s3://company-raw/events/")
)

clean_events = (
    events
    .withWatermark("event_ts", "30 minutes")
    .dropDuplicates(["event_id"])
)

query = (
    clean_events.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://company-checkpoints/events_silver/")
    .trigger(processingTime="5 minutes")
    .toTable("main.silver.events")
)

Real-world points:

+ Checkpoint locations are required for fault tolerance.
+ Watermarks define how late data can arrive before Spark drops state.
+ Deduplication needs a stable event key.
+ Streaming writes must be idempotent or carefully checkpointed.

How it works under the hood. A streaming query is a batch plan re-executed on every trigger. State for deduplication and aggregations lives in the state store under `checkpointLocation`, and a watermark bounds how long that state is kept. The driver keeps the query running and schedules one micro-batch job per trigger; the sink commits each batch transactionally.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
def merge_into_lakehouse(batch_df, batch_id):
    batch_df.createOrReplaceTempView("staged_events")
    spark.sql(
        """
        merge into main.silver.events as target
        using staged_events as source
        on target.event_id = source.event_id
        when matched then update set *
        when not matched then insert *
        """
    )

Worked example. A 5-minute tumbling window with a 30-minute watermark keeps at most about six windows of state per key; drop the watermark and the state store grows for the life of the query.

Common mistake: `outputMode("complete")` with an unbounded key space, which recomputes and rewrites the whole result on every batch. Use `append` or `update` with a watermark for streaming sinks.

## How do you monitor and debug PySpark jobs?

Start with the Spark UI. It shows jobs, stages, tasks, shuffle read/write, spilled memory, input size, executor failures, and SQL plans.

Useful PySpark checks:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
df.explain("formatted")
print(df.rdd.getNumPartitions())
spark.conf.get("spark.sql.shuffle.partitions")

Common symptoms:

+ One task runs much longer than others: likely skew
+ Huge shuffle read/write: expensive join, aggregation, or repartition
+ Spilled memory: insufficient executor memory or too-wide shuffle
+ Many tiny tasks: small files or too many partitions
+ Driver out of memory: collecting too much data to the driver

For production jobs, log record counts at each pipeline boundary:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
input_count = orders.count()
output_count = valid_orders.count()

spark.createDataFrame(
    [("orders_silver", input_count, output_count)],
    ["pipeline_name", "input_count", "output_count"],
).write.mode("append").saveAsTable("main.ops.pipeline_metrics")

For deeper performance methods, see [Spark Tuning](spark-tuning.md).

How it works under the hood. The driver's DAG scheduler tracks stages and tasks and emits the event log the Spark UI reads. The numbers that matter are per-stage: shuffle read and write bytes, spill to memory and disk, the task duration distribution, and GC time. A stage boundary exists wherever the plan contains an `Exchange`.

Worked example. If 199 tasks in a stage finish in 20 seconds and one takes 600 seconds, the stage wall time is 600 seconds. Adding partitions will not help; the hot key must be split with AQE skew join or manual salting.

Common mistake: looking only at the average task time. The max-to-median ratio is the skew signal, and excessive driver-side `collect()` shows up as driver GC, not as a slow task.

## How do you make a PySpark job faster?

Tuning starts with the plan and the Spark UI, not with a config change. Find the widest stage, read its shuffle bytes, spill and task-duration spread, then fix the one thing that dominates. The levers below are ordered by how often they matter. The deeper recipes live in [Spark Tuning](spark-tuning.ipynb).

**Adaptive Query Execution (AQE).** AQE re-optimizes the plan after shuffle stages report real sizes. It coalesces many small post-shuffle partitions, splits oversized partitions for skewed joins, and can switch a sort-merge join to a broadcast join at runtime. It is on by default since Spark 3.2.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")

**Shuffle partition sizing.** `spark.sql.shuffle.partitions` sets the reduce fan-out for every shuffle; the default 200 is wrong for both tiny and huge jobs. Aim for partitions of roughly 128-256 MB after the shuffle, and let AQE coalesce the small ones instead of hand-tuning per query.

In [ ]:
shuffle_bytes = 200 * 1024 ** 3
target_partition_bytes = 128 * 1024 ** 2
partitions = -(-shuffle_bytes // target_partition_bytes)  # ceiling division
print("target partitions:", partitions)

**Broadcast versus sort-merge join.** A broadcast join ships the small side to every executor and avoids shuffling the large side; it is chosen when the estimated side is under `spark.sql.autoBroadcastJoinThreshold` (10 MB by default). Fall back to sort-merge when neither side is small, and check the physical plan to confirm what was chosen.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql.functions import broadcast

threshold_bytes = int(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("broadcast threshold bytes:", threshold_bytes)

enriched = orders.join(broadcast(customers), on="customer_id", how="left")
enriched.explain("formatted")  # look for BroadcastHashJoin

**Skew.** One hot key puts most rows in one partition, so one task defines the stage duration. AQE skew join splits it when the partition exceeds both `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (256 MB) and 5x the median size. Otherwise, salt the hot key: add a random bucket to the fact side and explode the dimension across the same buckets.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

salt_buckets = 16
salted_facts = orders.withColumn("salt", (F.rand() * salt_buckets).cast("int"))
salted_dim = customers.withColumn(
    "salt", F.explode(F.array([F.lit(i) for i in range(salt_buckets)]))
)
joined = salted_facts.join(salted_dim, ["customer_id", "salt"])

**Caching.** Cache only a frame that several actions reuse or that feeds an iterative loop. Caching is lazy: the first action materialises the blocks, and you should `unpersist()` when done. A cached frame used once just adds memory pressure.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
enriched = orders.join(broadcast(customers), "customer_id").cache()
enriched.count()
enriched.groupBy("segment").count().show()
enriched.unpersist()

**File layout and predicate pushdown.** Partition on a low-cardinality column that appears in filters, target 100 MB-1 GB per file, and write a sensible number of files with `repartition` or `coalesce`. On the read side, `spark.sql.files.maxPartitionBytes` (128 MB) controls how much one task reads, and Parquet or Delta statistics skip files and row groups that cannot match the predicate.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

(
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .repartition("order_date")
    .write.mode("overwrite")
    .partitionBy("order_date")
    .parquet("s3://company-curated/orders/")
)

pruned = spark.read.parquet("s3://company-curated/orders/").filter("order_date = '2026-10-04'")
pruned.explain("formatted")  # PartitionFilters should list order_date

That is the short list. For the full treatment of AQE, partition sizing, join strategies, skew, storage layout and profiling, see [Spark Tuning](spark-tuning.ipynb).

## Part 5 - Real world and interview

Five use cases show where the patterns above pay off - nightly ELT, streaming ingestion,
feature tables, messy logs, safe backfills - then a section on how to say all of it in
an interview.


## Real world use cases for PySpark

### 1. Nightly batch ELT into a partitioned lake

Raw JSON lands in object storage, a Spark job cleans and types it, and the curated layer is written as partitioned Parquet for downstream engines.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders = spark.read.schema(order_schema).json("s3://acme-raw/orders/ds=2026-10-04/")

(
    orders.where("amount_cents > 0")
    .withColumn("amount", F.col("amount_cents") / 100)
    .repartition("order_date")            # one file family per day
    .sortWithinPartitions("country")      # tight min/max for skipping
    .write.mode("overwrite")
    .option("compression", "zstd")
    .partitionBy("order_date")
    .parquet("s3://acme-curated/orders/")
)

Why Spark: a few minutes of distributed CPU replaces hours of single-node scripting, and explicit schemas remove inference surprises. Enable Adaptive Query Execution (`spark.sql.adaptive.enabled=true`) so skewed joins and partition counts self-tune.

### 2. Structured Streaming from Kafka into a lakehouse table

A streaming job decodes Kafka events, applies the same transformations as the batch path, and merges micro-batches into an Iceberg or Delta table.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .writeStream
    .foreachBatch(merge_into_lakehouse)          # idempotent MERGE per batch
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)

Why Spark: the batch and streaming APIs share transformations, so there is one business-logic implementation. Checkpointing plus an idempotent merge gives end-to-end exactly-once.

### 3. Feature engineering for machine learning

Feature tables are built with broadcast joins to small dimensions and vectorized pandas UDFs instead of row-at-a-time Python UDFs.

In [ ]:
# ILLUSTRATIVE - requires pandas, pyspark: install to run, or read on.
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

features = (
    events.join(F.broadcast(products), "product_id")   # small dimension
    .groupBy("customer_id")
    .agg(F.count("*").alias("orders_30d"), F.sum("amount").alias("spend_30d"))
    .withColumn("spend_z", zscore("spend_30d"))
)

features.write.mode("overwrite").parquet("s3://acme-features/customer/ds=2026-10-04/")

Why Spark: the same feature definitions can be applied to historical training data and to daily scoring runs, which reduces train/serve skew.

### 4. Log and event processing with messy input

Semi-structured logs break schema assumptions. Read permissively, quarantine bad rows, and evolve the schema deliberately.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
raw = (
    spark.read.option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_bad")
    .json("s3://acme-raw/logs/ds=2026-10-04/")
)

good = raw.filter("_bad is null").drop("_bad")
bad = raw.filter("_bad is not null")

good.write.mode("append").parquet("s3://acme-curated/logs/")
bad.write.mode("append").json("s3://acme-quarantine/logs/")

Why Spark: quarantine plus schema-on-read lets ingestion continue while the platform team fixes the producer, instead of failing the whole day.

### 5. Backfills and reprocessing without duplicating data

A bug fix requires reprocessing September. With dynamic partition overwrite, only the partitions you touch are replaced.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

(
    reprocessed.withColumn("order_date", F.to_date("created_at"))
    .write.mode("overwrite")
    .partitionBy("order_date")
    .parquet("s3://acme-curated/orders/")
)

Why Spark: the same job that handles daily loads can reprocess a month, and dynamic overwrite makes the rerun idempotent at partition granularity - the property that makes backfills safe.

## What should you say in a PySpark interview answer?

A strong PySpark interview answer should include the API, the execution model, and the production concern.

Example answer for "How do you deduplicate orders?":

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import Window, functions as F

w = Window.partitionBy("order_id").orderBy(F.col("source_updated_at").desc())

deduped = (
    spark.table("main.bronze.orders")
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

Then explain:

+ `row_number` keeps one row per business key.
+ Ordering must use source metadata, not random ingestion order.
+ The result should be merged into the target for idempotency.
+ If one key is extremely hot, this window can skew and needs tuning.

That answer shows that you know PySpark syntax and the operational behavior behind it.

How it works under the hood, restated as an answer structure. A strong answer names the API, the execution model, and the failure mode. For deduplication: "I use `row_number()` over `Window.partitionBy(business_key).orderBy(source_lsn desc)` filtered to `rn = 1`, then `MERGE`; that forces one shuffle by key, and if one key is hot the window skews, so I check the partition sizes and salt or let AQE skew join split it."

Worked example, a 30-second answer skeleton:

```text
1. API       - the exact functions and window spec you call
2. Engine    - narrow vs wide, where the shuffle is, partition count
3. Ops       - idempotency, skew, cost, monitoring, failure recovery
```

Common mistake: answering with syntax only. Interviewers probe the plan, so tie every method to a shuffle, a partition count, or a failure mode.

## Where to run everything for real

The workbook below is the complete practice notebook, merged here so this notebook is
self-contained: its cells execute exactly the ideas above, with an assert cluster after
each one. Run it top to bottom now. Without PySpark and Java the cells print a SKIPPED
message, and the expected output sits in the markdown above each one.


---

# Practice notebook: PySpark concepts

The cells below are from `notebooks/04_pyspark_concepts.ipynb` and are merged here so this notebook is self-contained. Run them top to bottom; assertions confirm the expected behavior.

# 04 - PySpark Concepts, End to End

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*

A hands-on tour of the PySpark concepts that come up most often in data-engineering interviews.

The notebook is deliberately granular: one idea per cell, a short explanation directly above
every guarded code cell, and a small self-check block of `assert`s after each cluster of
operations. Every Spark section has four parts:

1. **Concept** - the mental model, in plain language.
2. **Guarded code** - runnable PySpark that only executes when Spark is really available.
3. **Expected output** - what you should see, so the notebook still teaches without Spark.
4. **Interview note** - the trade-off a strong candidate mentions out loud.

**How to run it**

1. Run the cells top to bottom with `Shift+Enter`, or use *Run All*.
2. Spark sections are guarded: without Java 17+ and PySpark they print a `SKIPPED` message instead of
   failing, and each **Expected output** block shows what the real run would print. Install them with
   `make setup-spark` to run everything for real.

## Prerequisites

- Notebook tooling: `uv sync --extra notebooks`
- Spark cells: `uv sync --extra notebooks --extra notebooks-spark`
- A **Java 17+** runtime on `PATH` (`java -version` must succeed).

PySpark talks to the JVM, so without Java `import pyspark` can succeed while every job fails.

## How to read this notebook without Spark

This machine may not have PySpark or Java installed. That is fine. The bootstrap cells probe
the environment and set `SPARK_AVAILABLE`. When it is `False`:

- every Spark cell prints `SKIPPED - install PySpark + Java 17 and re-run this cell`;
- the markdown **above** each Spark cell contains the expected output and the reasoning, so
  the concept is still fully covered;
- all `assert` statements live **inside** the Spark-available branch, so nothing fails just
  because Spark is missing.

## Concepts covered, in five parts

**Part 1 - Get a session and look at data.** SparkSession and local mode;
DataFrames from Python data with an explicit schema; lazy evaluation;
`select`/`filter`/`withColumn`/`rename`/`cast`.

**Part 2 - Shape the data.** Null handling; aggregations (`groupBy`/`agg`/`pivot`);
joins and broadcast hints; window functions; UDFs vs built-ins; arrays and `explode`.

**Part 3 - Run it like production.** Partitioning and Parquet partition pruning;
caching; `explain()` plan reading; AQE and skew; small files and compaction; a
Structured Streaming sketch; Delta and Iceberg notes on Databricks; platform translation.

**Part 4 - Sharpen the arithmetic without a cluster.** Eight standard-library exercises
that train the numbers behind Part 3.

**Part 5 - Answer out loud.** Fifteen interview questions covering the ground above.


## Running this notebook in a hosted session

A hosted interpreter (Colab, a managed runner, or a machine you upload the notebook to) usually ships a
Java runtime but not PySpark. In that case run one setup cell first:

```python
!pip install pyspark
```

Then restart the kernel and run all cells; the bootstrap cells below re-check the environment and
turn the Spark cells on. If PySpark is unavailable the notebook still runs: every Spark cell prints a
SKIPPED message and the markdown above it explains the expected result.


## Environment bootstrap: probe

**Expected output (no Spark here):**

```
PySpark installed : False
Java on PATH      : True
Java runnable     : False
Spark available   : False
Colab runtime     : False
```

`shutil.which("java")` only proves the binary exists: on some machines (e.g. macOS with only a
stub at `/usr/bin/java`) the executable is present but cannot start a JVM, so it is probed.

In [1]:
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ValueError):  # `google` is not installed outside Colab
    IS_COLAB = False

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(
            ["java", "-version"], capture_output=True, text=True, timeout=60
        )
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
print(f"Colab runtime     : {IS_COLAB}")
print()


PySpark installed : True
Java on PATH      : True
Java runnable     : False
Spark available   : False
Colab runtime     : False



**Expected output (no Spark here):**

```
This notebook will run in SKIPPED mode: every Spark cell prints a
SKIPPED message, and the markdown above each cell still teaches the concept.

To enable the Spark cells:
  1. uv sync --extra notebooks --extra notebooks-spark
  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)
  3. Make sure `java -version` works, then re-run this cell.
  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)
  - missing: pyspark (the Python package)
```

**Interview note:** "it imports fine" is not "it runs". A quick capability probe up front
saves an hour of debugging a `JAVA_HOME`/`SparkContext` stack trace later - the same habit you
want in a production job that must fail fast on a broken runtime.

In [2]:
if SPARK_AVAILABLE:
    import pyspark
    print(f"PySpark version   : {pyspark.__version__}")
    print("Spark cells below will execute for real.")
else:
    print("This notebook will run in SKIPPED mode: every Spark cell prints a")
    print("SKIPPED message, and the markdown above each cell still teaches the concept.")
    print()
    if IS_COLAB:
        print("Colab detected. To enable the Spark cells here:")
        print("  1. Add a cell containing:  !pip install pyspark")
        print("  2. Run it, then choose Runtime -> Restart session")
        print("  3. Run all cells again (Colab already provides Java).")
    else:
        print("To enable the Spark cells:")
        print("  1. uv sync --extra notebooks --extra notebooks-spark")
        print("  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)")
        print("  3. Make sure `java -version` works, then re-run this cell.")
    if not HAS_PYSPARK:
        print("  - missing: pyspark (the Python package)")
    if not HAS_JAVA:
        print("  - missing: a `java` executable on PATH")
    elif not JAVA_RUNS:
        print("  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)")


This notebook will run in SKIPPED mode: every Spark cell prints a
SKIPPED message, and the markdown above each cell still teaches the concept.

To enable the Spark cells:
  1. uv sync --extra notebooks --extra notebooks-spark
  2. Install a Java 17+ runtime (Temurin, or `brew install openjdk@17`)
  3. Make sure `java -version` works, then re-run this cell.
  - missing: a working Java 17+ runtime (the `java` on PATH cannot start)


## Part 1 - Get a session and look at data

Three moves open every PySpark notebook you will ever read: build a session, declare a
schema, transform lazily. Learn them in this order - every later part assumes you can
say what a transformation *will* do before it runs.


## 1. SparkSession and local mode

### Concept

`SparkSession` is the single entry point to Spark SQL and DataFrames. It wraps a
`SparkContext` (the low-level RDD scheduler) plus the SQL catalog and configuration.

### Why `local[*]`

`master("local[*]")` runs the whole cluster - driver and one executor - inside this JVM,
using one core per CPU for parallelism.

In a cluster you drop `.master(...)` and let the cluster manager (YARN, Kubernetes,
Databricks) supply it. Hard-coding a master is the classic bug that makes a job run on one
node in production.

Key configs used here:

- `spark.sql.shuffle.partitions` - partitions produced by a shuffle (default 200).
- `spark.ui.enabled=false` - skip the web UI in a notebook.

### Under the hood: driver, executors, and the DAG

A Spark application has two kinds of process.

- The **driver** runs `main()`, owns the `SparkSession` and `SparkContext`, turns your
  DataFrame chain into a logical plan, hands it to Catalyst, and then asks the **DAG
  scheduler** to cut the physical plan into **stages**. It also schedules tasks onto
  executors and collects the small results that actions return.
- **Executors** are long-lived JVM processes on the worker nodes. Each executor runs
  several tasks in parallel threads, holds cached blocks, and keeps the local shuffle
  output that the next stage reads.

A **stage** is a run of narrow transformations that can be pipelined without moving data
between executors. A **shuffle** ends the stage and starts a new one, because the next
stage cannot begin until every upstream task has written its shuffle files. That is why a
job appears as several stages in the Spark UI, and why the widest operation in a chain
usually defines the runtime.

In `local[*]` the driver and the single executor share one JVM, and `*` means "use every
available core as a task slot". You still get real task-level parallelism, but you never
see network shuffle, executor loss, or data locality, so local timings are a correctness
check, not a capacity plan.

### Common mistake: hard-coding the master and forking sessions

Copying `.master("local[*]")` into a job that runs on YARN, Kubernetes, or Databricks
pins the whole pipeline onto one machine. The cluster manager is supposed to supply the
master, so production code should leave it out and let `spark-submit` or the platform set
it.

The second habit to avoid is creating a session per function. `SparkSession.builder`
with `getOrCreate()` returns the existing active session, but building one in a library
that also sets conflicting configs produces confusing "multiple SparkContext" errors.
Create the session once, pass it down, and stop it only at the end of the process.

### Expected output (with Spark)

```
Spark version : 3.5.x
Master        : local[*]
App name      : de-interview-pyspark-concepts
```

### Interview note

Describe the driver/executor split here. The driver plans work and collects results; executors
run tasks and hold cached blocks. `local[*]` collapses both into one process, which is why it
is a dev tool, not a deployment topology.

In [3]:
if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("de-interview-pyspark-concepts")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")

    print("Spark version :", spark.version)
    print("Master        :", spark.sparkContext.master)
    print("App name      :", spark.sparkContext.appName)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Local mode in numbers

`defaultParallelism` is how many task slots this session can run at once. On
`local[*]` that is the number of cores, so the configured shuffle partition count divided
by the slot count is the number of waves of tasks. A large ratio is not automatically
bad - it usually beats starving the cores - but it is the first number to check when a
local run feels slow.

In [4]:
if SPARK_AVAILABLE:
    # Depth check: local[*] exposes one task slot per core.
    slots = spark.sparkContext.defaultParallelism
    shuffle_parts = int(spark.conf.get("spark.sql.shuffle.partitions"))
    print("task slots (cores)        :", slots)
    print("configured shuffle parts  :", shuffle_parts)
    print("waves if fully shuffled   :", -(-shuffle_parts // slots))
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 1 passed: local session, correct app name
```

**Check yourself.** The session must be local and named exactly as requested.

In [5]:
if SPARK_AVAILABLE:
    # --- self-check 1 ---------------------------------------------------
    assert spark.sparkContext.master.startswith("local"), spark.sparkContext.master
    assert spark.sparkContext.appName == "de-interview-pyspark-concepts"
    print("self-check 1 passed: local session, correct app name")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 2. DataFrames from Python data with an explicit schema

### Concept

`spark.createDataFrame(data, schema)` turns local Python data into a distributed DataFrame.

### Why an explicit schema

Passing an explicit `StructType` rather than letting Spark infer types is the production habit:
inference reads the data twice, guesses `string` for dates and numbers, and silently changes
nullability. Explicit schemas also document the contract and fail fast when the upstream shape
changes.

`LongType` / `DoubleType` / `StringType` map to `bigint` / `double` / `string` in the
DataFrame. The third field of `StructField` is `nullable`.

### Under the hood: Catalyst and why inference is a trap

`createDataFrame` walks a fixed pipeline: a logical plan is built first, Catalyst applies
rule-based rewrites (type coercion, nullability, constant folding), a cost-based
optimizer picks join and aggregate strategies, and the physical plan is code-generated by
Tungsten into compact JVM bytecode. Your explicit `StructType` is the contract that all of
that starts from.

Schema inference is a different code path: Spark reads the source once just to guess
types, then reads it again to build the frame. The guesses are data-dependent, so an empty
file yields an all-string schema, an integer column that happens to see one decimal
becomes `double`, and a column that is nullable today can be typed non-nullable tomorrow.
The inferred schema is also not stable across files with different shapes.

### Common mistake: letting the schema drift silently

The expensive failure is not a crash, it is a schema that changes between runs. A file
with a missing column, an extra column, or different casing can produce a frame that still
writes successfully with wrong content.

Declare the schema, and combine it with an explicit bad-record policy: `mode` =
`PERMISSIVE` (null out bad fields), `DROPMALFORMED` (drop the row), or `FAILFAST` (raise).
In a curated or silver layer, `FAILFAST` plus a quarantine path is usually right: you
would rather fail one batch and inspect the file than quietly lose rows. For casting
inside the frame, remember that a failed `cast` returns `null` rather than raising, so a
silent type problem looks like a sudden spike in nulls - another reason to assert row and
null counts after every load.

### Expected output (with Spark) - schema

```
root
 |-- order_id: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_ts: string (nullable = true)
```

### Expected output (with Spark) - the frame

```
+--------+-----------+------+---------+-------------------+
|order_id|customer_id|amount|status   |order_ts           |
+--------+-----------+------+---------+-------------------+
|1       |c1         |120.5 |paid     |2026-10-01 10:00:00|
|2       |c2         |80.0  |paid     |2026-10-01 11:30:00|
|3       |c3         |200.0 |shipped  |2026-10-02 09:15:00|
|4       |c1         |15.75 |cancelled|2026-10-02 12:00:00|
|5       |c4         |60.0  |paid     |2026-10-03 08:45:00|
|6       |c2         |30.0  |returned |2026-10-03 16:20:00|
+--------+-----------+------+---------+-------------------+
```

### Interview note

Mention schema enforcement and corrupt-record handling together:
`mode=PERMISSIVE/FAILFAST/DROPMALFORMED` on the reader, and why `FAILFAST` in a curated layer is
usually right. A schema inferred in dev and drifted in prod is a top cause of silent data
corruption.

In [6]:
if SPARK_AVAILABLE:
    from pyspark.sql.types import (
        DoubleType,
        LongType,
        StringType,
        StructField,
        StructType,
    )

    orders_schema = StructType([
        StructField("order_id", LongType(), False),
        StructField("customer_id", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("order_ts", StringType(), True),
    ])
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Building the rows

`orders_data` is a plain Python list of tuples; it is not distributed until
`createDataFrame` sees it.

**Expected output (with Spark):** the same six rows shown in the table above.

In [7]:
if SPARK_AVAILABLE:
    orders_data = [
        (1, "c1", 120.50, "paid", "2026-10-01 10:00:00"),
        (2, "c2", 80.00, "paid", "2026-10-01 11:30:00"),
        (3, "c3", 200.00, "shipped", "2026-10-02 09:15:00"),
        (4, "c1", 15.75, "cancelled", "2026-10-02 12:00:00"),
        (5, "c4", 60.00, "paid", "2026-10-03 08:45:00"),
        (6, "c2", 30.00, "returned", "2026-10-03 16:20:00"),
    ]

    orders = spark.createDataFrame(orders_data, schema=orders_schema)
    orders.printSchema()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Inspecting the frame

`show(truncate=False)` reprints the table without cutting long column values.

**Expected output (with Spark):** the six-row table above.

In [8]:
if SPARK_AVAILABLE:
    orders.show(truncate=False)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 2 passed: 6 rows, amount typed double, order_id non-nullable
```

**Check yourself.** Row count, the declared type of `amount`, and the nullability of
`order_id` are all part of the schema contract.

In [9]:
if SPARK_AVAILABLE:
    # --- self-check 2 ---------------------------------------------------
    assert orders.count() == 6
    assert orders.schema["amount"].dataType == DoubleType()
    assert orders.schema["order_id"].nullable is False
    print("self-check 2 passed: 6 rows, amount typed double, order_id non-nullable")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 3. Lazy evaluation: transformations vs actions

### Concept

A **transformation** (`filter`, `withColumn`, `select`, `join`, `groupBy`) only *describes* new
work and returns a new immutable DataFrame. Nothing is scheduled.

An **action** (`count`, `show`, `collect`, `write`, `take`, `foreach`) asks for a concrete
result; only then does Catalyst optimise the whole chain into stages and tasks and the
scheduler launch them.

### Why it matters

This is why a notebook cell that builds five DataFrames finishes instantly, and the cell with a
single `.count()` is the one that costs money. It is also why `withColumn` chains compile into
one scan rather than five passes.

### Under the hood: how an action becomes a job

Nothing in a transformation touches the cluster. Instead, each call appends a node to an
immutable logical plan. When you finally call an action, Catalyst analyses and optimises
the whole accumulated plan in one pass, the physical planner chooses operators, and the
DAG scheduler splits the result into stages and tasks. Only then are tasks serialised and
sent to executors.

Real work is lazy, so a DataFrame is a recipe, not a result. It also means the recipe can
be reused: building two frames from the same parent costs planning only, and the actual
scan happens once per action unless a shuffle or a cache materialised something in
between.

### Common mistake: measuring a lazy object, and re-running actions

`print(df)` and `df.explain()` execute nothing, so a notebook can look fast while being
untested. The opposite mistake is calling several actions on the same uncached frame:
`df.count()` for a log line, `df.show()` for a look, `df.write` for the sink - each one
re-reads the source and recomputes the entire chain. On a big table that is three scans
where one was needed. If you genuinely need the intermediate more than once, cache it
(the caching section in Part 3) or write it once and read it back.

### Expected output (with Spark)

```
After two transformations, no job has been submitted yet.
isStreaming = False
paid rows: 3
```

### Interview note

The follow-up is always "so when does a job run?". Answer: at the action, and one action = one
job = one fresh pass over the source unless you cached or reused a shuffled/saved result.

In [10]:
if SPARK_AVAILABLE:
    from pyspark.sql import functions as F

    # Transformation 1 - lazy.
    paid_only = orders.filter(F.col("status") == "paid")
    # Transformation 2 - still lazy, still nothing executed.
    with_tax = paid_only.withColumn("amount_with_tax", F.round(F.col("amount") * 1.08, 2))

    print("After two transformations, no job has been submitted yet.")
    print("isStreaming =", with_tax.isStreaming)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Reading the plan without running it

`explain(True)` prints the parsed, analysed and optimised plans; it reads no data.

**Expected output (with Spark):** a logical/physical plan tree for the filter-plus-withColumn
chain.

In [11]:
if SPARK_AVAILABLE:
    with_tax.explain(True)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### The action is what triggers a job

**Expected output (with Spark):**

```
paid rows: 3
```

In [12]:
if SPARK_AVAILABLE:
    paid_rows = with_tax.count()  # <-- action: this is what triggers a job
    print("paid rows:", paid_rows)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 1b passed: the action returned 3 paid rows
```

**Check yourself.** The count only exists because the action above ran.

In [13]:
if SPARK_AVAILABLE:
    assert paid_rows == 3
    print("self-check 1b passed: the action returned 3 paid rows")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 4. `select`, `filter`, `withColumn`, `rename`, `cast`

### Concept

These are the everyday column transformations and all are **narrow** (no data exchange between
executors):

- `select` / `drop` choose columns - push the projection down so Parquet reads fewer bytes.
- `filter` / `where` are aliases; predicates are pushed to the scan when possible.
- `withColumn` replaces or adds a column from an expression.
- `withColumnRenamed` renames; `alias` does it inline in a `select`.
- `cast` changes the type; do it explicitly and early, and know that a bad cast yields `null`
  (masking errors) instead of raising.

### Under the hood: narrow work, pushdown, and the plan shape

`select`, `filter`, `withColumn`, `cast`, and `drop` are narrow: each output partition
depends on exactly one input partition, so Spark pipelines them into a single stage with
no network exchange. Catalyst then pushes as much as it can down toward the scan:
projections reduce the columns Parquet decodes, and predicates that only touch source
columns become `PushedFilters` on the file scan. Pushdown is why a filtered `select` on a
wide Parquet table can read a fraction of the bytes instead of the whole file.

Tungsten's whole-stage code generation fuses these operators into one generated function
per stage, so the per-row overhead of the DataFrame API largely disappears. A chain of ten
narrow operations is not ten passes; it is one generated loop over the data.

### Common mistake: withColumn in a loop, and casts that hide errors

Every `withColumn` adds a projection node to the plan. Calling it inside a Python loop
over a hundred columns produces a hundred-deep plan that Catalyst must analyse and that
can blow up driver memory or hit plan-size limits. Build the columns in one `select` (or
one `withColumns` on newer Spark) instead.

The other trap is `cast`. It never raises on bad input; it returns `null`. So
`col("amount").cast("double")` on a dirty string column turns corruption into missing
data. If the cast matters, validate the null rate before and after, or guard it with
`when`, and keep the raw string column until the conversion is proven.

### Expected output (with Spark)

```
root
 |-- order_id: long (nullable = false)
 |-- cust_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- amount_cents: long (nullable = true)

4 rows, columns: ['order_id', 'cust_id', 'amount', 'order_ts', 'amount_cents']
amount_cents first row = 12050
```

### Interview note

Say "narrow transformation" and explain why it matters: narrow steps stay inside a partition,
so they need no shuffle. Wide steps (`groupBy`, `join`, `distinct`, `repartition`, window with a
different partitioning) force an exchange and dominate cost.

### Step 1 - project and filter

**Expected output (with Spark):** four rows, the union of statuses `paid` and `shipped`.

In [14]:
if SPARK_AVAILABLE:
    selected = (
        orders
        .select("order_id", "customer_id", "amount", "status", "order_ts")
        .filter(F.col("status").isin("paid", "shipped"))
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Step 2 - derive and cast columns

`to_timestamp` parses the string into a `timestamp`; multiplying by 100 and casting to `long`
turns dollars into integer cents.

In [15]:
if SPARK_AVAILABLE:
    derived = (
        selected
        .withColumn("order_ts", F.to_timestamp("order_ts"))
        .withColumn("amount_cents", (F.col("amount") * 100).cast("long"))
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Step 3 - rename and drop

`withColumnRenamed("customer_id", "cust_id")` renames in place; `drop("status")` removes a
column. Both keep the frame narrow.

In [16]:
if SPARK_AVAILABLE:
    transformed = derived.withColumnRenamed("customer_id", "cust_id").drop("status")
    transformed.printSchema()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Evaluate

**Expected output (with Spark):**

```
4 rows, columns: ['order_id', 'cust_id', 'amount', 'order_ts', 'amount_cents']
amount_cents first row = 12050
```

In [17]:
if SPARK_AVAILABLE:
    rows = transformed.collect()
    print(len(rows), "rows, columns:", transformed.columns)
    print("amount_cents first row =", rows[0]["amount_cents"])
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 3 passed: types, rename and row count all correct
```

**Check yourself.** `count()` is an action, so this cell also materialises the chain.

In [18]:
if SPARK_AVAILABLE:
    # --- self-check 3 ---------------------------------------------------
    assert transformed.count() == 4  # paid x3 + shipped x1
    assert "cust_id" in transformed.columns and "customer_id" not in transformed.columns
    assert dict(transformed.dtypes)["amount_cents"] == "bigint"
    assert dict(transformed.dtypes)["order_ts"] == "timestamp"
    print("self-check 3 passed: types, rename and row count all correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## Part 2 - Shape the data

Part 1 gave you a session and a schema. Now make rows and columns do what the business
wants: clean nulls, aggregate, join, rank with windows, and reach for Python only when
the built-ins cannot express it. This is where pipelines spend their runtime, and where
interviewers press hardest.


## 5. Null handling

### Concept

Spark SQL is null-aware: `null == null` is `null` (not true), so use `isNull` / `isNotNull` /
`eqNullSafe` (`<=>`) rather than `==`.

`F.col("x") == None` does **not** work - the comparison yields null and filters everything.

### The toolkit

- `isNull()`, `isNotNull()` for predicates;
- `na.drop(subset=[...])` to remove rows, `na.fill({...})` to substitute values;
- `coalesce(a, b, lit(default))` to pick the first non-null expression;
- `when(...).otherwise(...)` for conditional logic (handles null explicitly);
- `nullif` to turn a sentinel value back into a null.

### Under the hood: three-valued logic

Spark SQL follows SQL null semantics, which is three-valued logic. A comparison with a
null operand yields `unknown`, not `false`, and `filter` keeps only rows where the
predicate is definitely true. That is why `col("x") == None` filters everything: it
evaluates to `unknown` for every row. `isNull`, `isNotNull`, and the null-safe equals
`eqNullSafe` (`<=>`) exist precisely because `=`, `<`, and `+` propagate nulls.

The same rule governs expressions: `null + 1` is `null`, `coalesce` stops at the first
non-null argument, and `when` treats an `unknown` condition as "not matched", so a branch
with no `otherwise` yields `null`. Aggregates have their own conventions: `count(*)`
counts rows, `count(col)` counts non-null values, `sum` of all nulls is null, and `avg`
divides by the non-null count only.

### Common mistake: treating null and empty as the same thing

Null, empty string, `0`, and `NaN` are four different values, and pipelines that merge
them lose information. Concretely:

- `na.fill({"amount": 0.0})` turns "we do not know the amount" into "the amount is zero",
  which changes every `avg` and `sum` downstream. Fill only when the default is meaningful.
- `filter(col("x") == "")` does not match nulls, and `isNull()` does not match empty
  strings; if a broken event writes empty strings, the null check misses it.
- `dropna()` with no arguments drops any row containing any null, which can delete far
  more than intended. Prefer `subset=[...]` and say which keys are actually required.

Decide the semantics before writing the code: is a missing tag "unknown" (null) or "none"
(empty array)? That one decision is the difference between a correct and a silently
under-counting pipeline.

### Expected output (with Spark)

```
raw rows: 4
rows with any of amount/status null: 3
after fill+drop: 3 rows, 0 null amounts, 0 null statuses
coalesce produced 2 'unknown' statuses
```


In [19]:
if SPARK_AVAILABLE:
    nulls = spark.createDataFrame(
        [
            (1, "c1", 120.5, None),
            (2, "c2", None, "paid"),
            (3, None, 50.0, "paid"),
            (4, "c4", 0.0, None),
        ],
        "order_id long, customer_id string, amount double, status string",
    )
    nulls.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Counting the holes

**Expected output (with Spark):**

```
raw rows: 4
rows with any of amount/status null: 3
```

In [20]:
if SPARK_AVAILABLE:
    missing = nulls.filter(F.col("amount").isNull() | F.col("status").isNull())
    print("raw rows:", nulls.count())
    print("rows with any of amount/status null:", missing.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Fill and drop

`na.fill` substitutes values for nulls, then `na.drop(subset=[...])` removes rows whose key
column is still null.

**Expected output (with Spark):**

```
after fill+drop: 3 rows, 0 null amounts, 0 null statuses
```

In [21]:
if SPARK_AVAILABLE:
    filled = nulls.na.fill({"amount": 0.0, "status": "unknown"}).na.drop(subset=["customer_id"])
    filled.show()
    print(
        "after fill+drop:",
        filled.count(),
        "rows,",
        filled.filter(F.col("amount").isNull()).count(),
        "null amounts,",
        filled.filter(F.col("status").isNull()).count(),
        "null statuses",
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### `coalesce` for a default

`coalesce` returns the first non-null argument, so it is the clean way to supply a fallback.

**Expected output (with Spark):**

```
coalesce produced 2 'unknown' statuses
```

In [22]:
if SPARK_AVAILABLE:
    coalesced = nulls.withColumn("status_clean", F.coalesce(F.col("status"), F.lit("unknown")))
    print("coalesce produced", coalesced.filter(F.col("status_clean") == "unknown").count(), "'unknown' statuses")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Interview note

Tie nulls to the metric. `avg` ignores nulls (denominator excludes them), `sum` of all-null is
null while `sum` with 0s is 0, and `count(col)` skips nulls while `count(*)` does not.
Interviewers use exactly this to test whether you trust a dashboard number.


### Expected output (with Spark)

```
self-check 4 passed: null semantics, fill/drop and coalesce all correct
```

In [23]:
if SPARK_AVAILABLE:
    # --- self-check 4 ---------------------------------------------------
    assert missing.count() == 3
    assert filled.count() == 3
    assert filled.filter(F.col("amount").isNull()).count() == 0
    assert filled.filter(F.col("status").isNull()).count() == 0
    assert coalesced.filter(F.col("status_clean") == "unknown").count() == 2
    print("self-check 4 passed: null semantics, fill/drop and coalesce all correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 6. Aggregations: `groupBy` / `agg` / `pivot`

### Concept

`groupBy(...).agg(...)` is a **wide** transformation: rows with the same key must land in the
same partition, so Spark performs a shuffle (a hash exchange) unless the data is already
co-partitioned.

### What a shuffle costs

Costs to name in an interview: shuffle write + read, disk spill, and the
`spark.sql.shuffle.partitions` fan-out.


### Under the hood: what a shuffle actually does

`groupBy(...).agg(...)` cannot run row by row, because rows with the same key start in
different partitions. Spark performs an **exchange**: every task writes its rows into one
file per output partition after hashing the key, the executors fetch the blocks they own
from the other executors (the network transfer), and the reducer-side tasks merge and
aggregate them.

Catalyst softens the cost with a map-side combine: `partial_sum` / `partial_count` run
before the exchange so each partition ships one row per key instead of every row. That is
why the plan shows `HashAggregate` on both sides of an `Exchange`. The shuffle still pays
to write intermediate data to local disk, transfer it, and possibly spill it when a
partition is larger than memory. The number of output partitions is fixed by
`spark.sql.shuffle.partitions` (default 200), independent of the input size.

### Common mistake: leaving shuffle partitions at the default

Two hundred partitions is a reasonable default for a medium job and a bad choice at both
extremes. On a small dataset it creates two hundred mostly empty tasks, each with scheduler
overhead and (if it is a write) a tiny output file. On a large dataset each partition can
be gigabytes, so tasks spill and a straggler defines the runtime.

Pick the count from the data, or let AQE do it: with AQE coalescing enabled, Spark starts
from the configured count and merges small post-shuffle partitions up to the advisory
size. The second mistake is pivoting on a high-cardinality column: `pivot` materialises
one output column per distinct value, so a column with ten thousand values produces a plan
with ten thousand columns. Always pass an explicit value list and treat pivot as a
presentation step, not a data-shaping default.

### Expected output (with Spark)

```
+------+-------+------+-------+
|region|revenue|orders|avg_qty|
+------+-------+------+-------+
|north |105.0  |2     |6.5    |
|south |125.0  |2     |6.0    |
+------+-------+------+-------+

+------+------+------+
|region|gadget|widget|
+------+------+------+
|north |30.0  |25.0  |
|south |70.0  |12.5  |
+------+------+------+
```

The pivot table is revenue by product: widget = 10*2.5 = 25 (north) and 5*2.5 = 12.5 (south);
gadget = 3*10 = 30 (north) and 7*10 = 70 (south).

### Build the input

**Expected output (with Spark):** a four-row frame of sales.


In [24]:
if SPARK_AVAILABLE:
    sales = spark.createDataFrame(
        [
            ("2026-10-01", "north", "widget", 10, 2.5),
            ("2026-10-01", "south", "widget", 5, 2.5),
            ("2026-10-02", "north", "gadget", 3, 10.0),
            ("2026-10-02", "south", "gadget", 7, 10.0),
        ],
        "sale_date string, region string, product string, qty int, unit_price double",
    )
    sales.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### `groupBy().agg()`

Three aggregates in one pass: a sum, a count and an average.

**Expected output (with Spark):** the `per_region` table above.

In [25]:
if SPARK_AVAILABLE:
    per_region = (
        sales
        .withColumn("revenue", F.col("qty") * F.col("unit_price"))
        .groupBy("region")
        .agg(
            F.sum("revenue").alias("revenue"),
            F.count("*").alias("orders"),
            F.avg("qty").alias("avg_qty"),
        )
    )
    per_region.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Pivot

`pivot` rotates a low-cardinality column into columns. It must first buffer the distinct pivot
values, so Spark asks you to supply them explicitly when you know them -
`.pivot("product", ["gadget", "widget"])` - which avoids an extra pass over the data.


### `pivot`

An explicit pivot-value list (`["gadget", "widget"]`) avoids the extra pass that Spark would
otherwise need to discover the distinct values.

**Expected output (with Spark):** the `region/gadget/widget` table above.

In [26]:
if SPARK_AVAILABLE:
    pivoted = (
        sales
        .withColumn("revenue", F.col("qty") * F.col("unit_price"))
        .groupBy("region")
        .pivot("product", ["gadget", "widget"])
        .agg(F.sum("revenue"))
    )
    pivoted.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Interview note

Follow up `groupBy` with partition sizing. Too few post-shuffle partitions and one task becomes
the bottleneck (skew); too many and you pay scheduler and small-file overhead. AQE can coalesce
them for you - see section 14.


### Expected output (with Spark)

```
self-check 5 passed: aggregation types and pivot values correct
```

In [27]:
if SPARK_AVAILABLE:
    # --- self-check 5 ---------------------------------------------------
    assert per_region.count() == 2
    assert dict(per_region.dtypes)["revenue"] == "double"
    assert set(pivoted.columns) == {"region", "gadget", "widget"}
    assert pivoted.filter(F.col("region") == "south").collect()[0]["gadget"] == 70.0
    print("self-check 5 passed: aggregation types and pivot values correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 7. Joins: inner, left, broadcast hints, and why shuffle happens

### Concept

A join keys two datasets on a common column. `how=` selects the semantics:

- `inner` keeps matches only;
- `left` keeps every left row and null-fills the right side;
- `left_anti` keeps left rows with **no** match (great for orphan detection);
- `left_semi` keeps left rows that match (great for existence checks).

### Why a shuffle?

In a standard sort-merge/hash join, matching keys must be in the same partition, so Spark
repartitions both sides by the join key: a full network exchange.

### Under the hood: how Spark chooses a join strategy

Spark has three main physical joins, and the planner picks between them.

- **Broadcast hash join**: if one side fits under `autoBroadcastJoinThreshold`
  (10 MB by default), the driver collects it, ships it to every executor, and builds a
  hash table in memory. The large side is never shuffled. This is the fastest strategy
  when it applies.
- **Sort-merge join**: both sides are exchanged by the join key, sorted, and merged. This
  is the general fallback, and it pays two shuffles plus the sort.
- **Shuffle hash join**: the smaller side of the shuffle is built into a hash table per
  partition. It avoids the sort but needs the build side to fit in memory.

Skew is what breaks all three. If one key holds a large share of the rows, the task that
receives that key does the work of many tasks, no matter how even the partitions look on
paper.

### Common mistake: broadcasting something that is not small

A broadcast hint is a memory promise. Every executor keeps a full copy of the broadcast
side, and in `local[*]` that copy lives in the driver JVM, so a "small" 500 MB dimension
table still risks an out-of-memory error. Hints that exceed the threshold are logged and
ignored rather than silently honoured, so always check `explain()` for
`BroadcastHashJoin` instead of assuming the hint took effect.

The other classic join bug is a type mismatch on the key - `int` on one side and `string`
on the other. Spark will implicitly cast and can still join, but the cast can defeat a
broadcast or produce null keys. Cast both sides explicitly before the join, and watch for
null keys: in an inner join they simply disappear, but a null join key can also pile every
unmatched row into one reducer.

### Build the inputs

`customers` is the small dimension; `small_orders` has one orphan row (`c9`).


In [28]:
if SPARK_AVAILABLE:
    customers = spark.createDataFrame(
        [("c1", "Ada"), ("c2", "Grace"), ("c3", "Linus")],
        "customer_id string, name string",
    )
    small_orders = spark.createDataFrame(
        [("o1", "c1", 10.0), ("o2", "c2", 20.0), ("o3", "c9", 5.0)],
        "order_id string, customer_id string, amount double",
    )
    customers.show()
    small_orders.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Inner and left joins

**Expected output (with Spark):**

```
inner rows: 2   (o3 has customer c9, which is not in customers)
left rows : 3
```

In [29]:
if SPARK_AVAILABLE:
    inner = small_orders.join(customers, on="customer_id", how="inner")
    left = small_orders.join(customers, on="customer_id", how="left")

    print("inner rows:", inner.count(), "  (o3 has customer c9, which is not in customers)")
    print("left rows :", left.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Finding the orphan

`left` null-fills the unmatched right side, so a null `name` marks the orphan.

**Expected output (with Spark):**

```
left rows with no matching customer: 1
```

In [30]:
if SPARK_AVAILABLE:
    print("left rows with no matching customer:", left.filter(F.col("name").isNull()).count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Broadcast join

If one side is small (default limit `spark.sql.autoBroadcastJoinThreshold` = 10 MB,
configurable), Spark ships that whole side to every executor and joins locally - no shuffle of
the big table. You force it with `F.broadcast(small_df)`, and the hint is ignored if the side
exceeds the threshold. `left_anti` also avoids the wide shuffle when it can be turned into a
broadcast anti-join.


### Broadcast hint

`broadcast(customers)` forces the small side to be shipped to every executor.
`explain()` should show `BroadcastHashJoin` instead of `SortMergeJoin`.

**Expected output (with Spark):**

```
broadcast join rows: 3
```

In [31]:
if SPARK_AVAILABLE:
    from pyspark.sql.functions import broadcast

    with_hint = small_orders.join(broadcast(customers), on="customer_id", how="left")
    with_hint.explain()  # look for BroadcastHashJoin in the physical plan
    print("broadcast join rows:", with_hint.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Compare the two join plans

Disabling auto-broadcast forces the planner onto a shuffle join; re-enabling it and adding
a hint gives the broadcast join back. Comparing the two physical plans is the clearest way
to see that a join strategy is a plan choice, not a syntax difference.

In [32]:
if SPARK_AVAILABLE:
    # Force the planner away from broadcast and inspect what it chooses instead.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    shuffled = small_orders.join(customers, on="customer_id", how="inner")
    shuffled_plan = shuffled._jdf.queryExecution().executedPlan().toString()
    assert "BroadcastHashJoin" not in shuffled_plan, shuffled_plan
    print("auto-broadcast off -> shuffle join in the plan")

    # Restore the default and hint the small side explicitly.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
    broadcast_join = small_orders.join(F.broadcast(customers), on="customer_id", how="inner")
    broadcast_plan = broadcast_join._jdf.queryExecution().executedPlan().toString()
    assert "BroadcastHashJoin" in broadcast_plan, broadcast_plan
    print("hint on            -> BroadcastHashJoin in the plan")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### `left_anti` for set difference

`left_anti` finds orphans without materialising right-hand columns.

**Expected output (with Spark):** one row, `o3`.

In [33]:
if SPARK_AVAILABLE:
    orphans = small_orders.join(customers, on="customer_id", how="left_anti")
    orphans.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
inner rows: 2   (o3 has customer c9, which is not in customers)
left rows : 3
left rows with no matching customer: 1
broadcast join rows: 3
```


### Expected output (with Spark)

```
self-check 6 passed: join semantics and broadcast hint correct
```

In [34]:
if SPARK_AVAILABLE:
    # --- self-check 6 ---------------------------------------------------
    assert inner.count() == 2
    assert left.count() == 3
    assert left.filter(F.col("name").isNull()).count() == 1
    assert with_hint.count() == 3
    assert orphans.collect()[0]["order_id"] == "o3"
    print("self-check 6 passed: join semantics and broadcast hint correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Interview note

The broadcast limit is about memory, not just time. Every executor must hold the small side; a
4 GB "small" dimension table in `local[*]` or on a small executor causes OOM. Also mention that
a **skewed** join key (one key = 40% of rows) defeats the shuffle join regardless of sizing -
the fix is salting or AQE skew join.


## 8. Window functions

### Concept

A window function computes over a set of rows related to the current row **without collapsing
them** the way `groupBy` does. You define a `Window`:

- `partitionBy(...)` - the groups (this is what forces a shuffle);
- `orderBy(...)` - the ordering inside each group;
- `rowsBetween(...)` / `rangeBetween(...)` - the frame for aggregations.

### The four workhorses

- `row_number()` - unique 1..n per partition; the standard **dedupe** idiom (`rn = 1` keeps
  exactly one row per key even for exact ties).
- `rank()` / `dense_rank()` - leave gaps / do not leave gaps on ties, so ties produce the same
  rank and more than one "rank 1".
- `sum(...).over(unboundedPreceding..currentRow)` - running total.
- `lag()` / `lead()` - previous / next value, e.g. time between events.

### Under the hood: how a window is executed

A window function keeps every input row, so unlike `groupBy` it cannot collapse the data.
Spark executes it as a `Window` node, which needs the input sorted within each partition
by the window's `partitionBy` and `orderBy` keys. That means a shuffle when `partitionBy`
is present, followed by a sort inside each partition.

The frame then decides which rows each output row sees. `rowsBetween` counts physical rows
and is cheap; `rangeBetween` compares values of the ordering column, so ties share the
frame and more rows must be buffered. When the frame runs from `unboundedPreceding` to
`currentRow`, Spark can compute the aggregate in one pass over the sorted partition and
emit a running value; peer-based or offset frames may need to buffer a window of rows.
`row_number`, `rank`, and `dense_rank` are rank operators, not aggregates, and need only
the ordering.

A window with no `partitionBy` sends every row to a single partition, so one task does all
the work. That is often the slowest line in an otherwise reasonable job.

### Common mistake: a non-deterministic "latest row" and a missing partition key

The dedupe idiom `row_number() over (partition by key order by updated_at desc)` is only
deterministic if `updated_at` is unique per key. When two rows share the maximum
timestamp, the tie is broken by whatever order the shuffle happened to produce, so the
winner can change between runs. Add a deterministic second sort key (a monotonically
increasing id, or a row hash) and the result becomes reproducible.

The second mistake is forgetting `partitionBy` and accidentally computing a global rank.
That forces all rows through one task, and it also changes the meaning of the query:
`row_number()` without a partition key ranks the entire dataset, not the group. If the
intent is a global rank, say so out loud; if it is per entity, the partition key is
mandatory.

### Expected output (with Spark)

```
row_number dedupe keeps: 3 rows (one per customer)
c2 winner amount: 99.0
rows with rank() == 1     : 4   (the tied pair shares rank 1)
c2 rank() values: [1, 1, 3]   dense_rank() values: [1, 1, 2]
running total for c1: 175.0
rows where lag() is null: 3   (one first row per customer)
```

Note the tie: `rank()` jumps to 3 after the tied pair, `dense_rank()` continues at 2. That is
the difference interviewers probe for.


### Build the events

`c2` has an exact duplicate row, which creates a real tie for `rank()`.

**Expected output (with Spark):** the seven-row frame.

In [35]:
if SPARK_AVAILABLE:
    from pyspark.sql.window import Window

    events = spark.createDataFrame(
        [
            ("c1", "2026-10-01", 100.0),
            ("c1", "2026-10-03", 50.0),
            ("c1", "2026-10-02", 25.0),
            ("c2", "2026-10-01", 10.0),
            ("c2", "2026-10-01", 99.0),
            ("c2", "2026-10-01", 99.0),  # exact duplicate -> a real tie
            ("c3", "2026-10-05", 7.0),
        ],
        "customer_id string, order_date string, amount double",
    )
    events.show()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### The window spec

Latest first, and `amount` desc as a deterministic tie-break so the "latest" row never depends
on run order.

In [36]:
if SPARK_AVAILABLE:
    by_customer_latest = Window.partitionBy("customer_id").orderBy(
        F.col("order_date").desc(), F.col("amount").desc()
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Dedupe with `row_number`

**Expected output (with Spark):**

```
row_number dedupe keeps: 3 rows (one per customer)
c2 winner amount: 99.0
```

In [37]:
if SPARK_AVAILABLE:
    deduped = (
        events
        .withColumn("rn", F.row_number().over(by_customer_latest))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
    print("row_number dedupe keeps:", deduped.count(), "rows (one per customer)")
    c2_amount = deduped.filter(F.col("customer_id") == "c2").collect()[0]["amount"]
    print("c2 winner amount:", c2_amount)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### `rank()` vs `dense_rank()`

**Expected output (with Spark):**

```
rows with rank() == 1     : 4   (the tied pair shares rank 1)
c2 rank() values: [1, 1, 3]   dense_rank() values: [1, 1, 2]
```

In [38]:
if SPARK_AVAILABLE:
    ranked = events.withColumn("rk", F.rank().over(by_customer_latest))
    dense = events.withColumn("drk", F.dense_rank().over(by_customer_latest))
    print("rows with rank() == 1     :", ranked.filter(F.col("rk") == 1).count(), "  (the tied pair shares rank 1)")
    c2_ranks = [r["rk"] for r in ranked.filter(F.col("customer_id") == "c2").orderBy("rk").collect()]
    c2_dense = [r["drk"] for r in dense.filter(F.col("customer_id") == "c2").orderBy("drk").collect()]
    print("c2 rank() values:", c2_ranks, "  dense_rank() values:", c2_dense)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Running total

A frame from `unboundedPreceding` to `currentRow` turns `sum` into a running total.

**Expected output (with Spark):**

```
running total for c1: 175.0
```

In [39]:
if SPARK_AVAILABLE:
    running_window = (
        Window.partitionBy("customer_id")
        .orderBy("order_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    running = events.withColumn("running_amount", F.sum("amount").over(running_window))
    c1_total = running.filter(F.col("customer_id") == "c1").agg(F.max("running_amount")).collect()[0][0]
    print("running total for c1:", c1_total)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### `lag` for the previous value

**Expected output (with Spark):**

```
rows where lag() is null: 3   (one first row per customer)
```

In [40]:
if SPARK_AVAILABLE:
    lagged = events.withColumn(
        "prev_amount",
        F.lag("amount").over(Window.partitionBy("customer_id").orderBy("order_date")),
    )
    print("rows where lag() is null:", lagged.filter(F.col("prev_amount").isNull()).count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Interview note

"dedupe with `row_number` over the natural key ordered by `updated_at desc`, then `rn = 1`" is
the canonical CDC/upsert answer. Mention the tie-break: put a deterministic second sort key in
the `orderBy`, or the "latest" row is non-deterministic across runs.


### Expected output (with Spark)

```
self-check 7 passed: dedupe, rank tie semantics, running total, lag
```

In [41]:
if SPARK_AVAILABLE:
    # --- self-check 7 ---------------------------------------------------
    assert deduped.count() == 3
    assert c2_amount == 99.0
    assert ranked.filter(F.col("rk") == 1).count() == 4
    assert c2_ranks == [1, 1, 3]
    assert c2_dense == [1, 1, 2]
    assert c1_total == 175.0
    assert lagged.filter(F.col("prev_amount").isNull()).count() == 3
    print("self-check 7 passed: dedupe, rank tie semantics, running total, lag")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 9. UDFs vs built-in functions

### Concept

A Python **UDF** ships a serialised Python function to each executor task, runs it **row by
row**, and converts every value between the JVM and the Python worker. That breaks Catalyst's
ability to optimise the expression and adds per-row pickling overhead - often 2-10x slower than
an equivalent built-in, and it blocks whole-stage code generation (Tungsten).

### Rules of thumb

1. Prefer `pyspark.sql.functions` (`when`, `regexp_extract`, `to_date`, `array_*`, ...).
2. If you truly need custom logic, prefer a **pandas UDF** (`@pandas_udf`) so work is vectorised
   per batch, or `F.udf` only for small, non-hot paths.
3. Never do network/DB calls inside a UDF - that is per-row I/O on every executor.

The example below computes the same tier with both approaches. Both give identical results; the
built-in is the one you should ship.

### Under the hood: why a Python UDF breaks the fast path

Normally Catalyst compiles a DataFrame expression into JVM bytecode. Tungsten then
generates one function per stage (whole-stage code generation) and processes rows in a
compact binary format, with no Python involved. A `F.udf` punctures that pipeline: each
executor starts a Python worker process, pickles every input value to it, runs your
function in the interpreter, and pickles the result back. The optimizer sees an opaque
`PythonUDF` node, so it cannot push predicates through it, fold constants, or
code-generate the stage.

The cost is per row, not per byte, which is why the slowdown is worst on wide tables with
small rows. A **pandas UDF** (`@pandas_udf`) is the middle ground: Apache Arrow batches
columns between the JVM and Python, so the interpreter overhead is amortised over a batch
and vectorised libraries such as NumPy or pandas can do the work. On Databricks, a Scala
or Java UDF gives JVM-native execution with no bridge at all.

### Common mistake: reaching for a UDF before checking the built-ins

Most "I need a UDF" cases have a built-in: `when`/`otherwise` for branching,
`regexp_extract`/`regexp_replace` for text, `to_date`/`date_add` for dates,
`transform`/`filter`/`aggregate` for arrays, and `try_cast`/`try_divide` for error-safe
math. Higher-order functions can express loops over arrays without dropping to Python.

When a UDF really is needed, keep it pure and stateless. A UDF that opens a database
connection is per-row network I/O multiplied by every task; a UDF that closes over a large
object ships that object with the closure to every worker. Do the lookup as a join against
a small dimension table instead, and reserve Python UDFs for cold paths or one-off
cleanup.

### Expected output (with Spark)

```
+-------+---------+
|amount |tier_udf |
+-------+---------+
|120.5  |high     |
|80.0   |mid      |
|200.0  |high     |
|15.75  |low      |
|60.0   |mid      |
|30.0   |low      |
+-------+---------+

UDF 'high' rows   : 2
Built-in 'high' rows: 2
```

### Define the UDF

**Expected output (with Spark):** nothing yet - the UDF is only registered here.


In [42]:
if SPARK_AVAILABLE:
    from pyspark.sql.types import StringType

    def _tier(amount):
        if amount is None:
            return "unknown"
        if amount >= 100:
            return "high"
        if amount >= 50:
            return "mid"
        return "low"

    tier_udf = F.udf(_tier, StringType())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### The built-in equivalent

`when(...).otherwise(...)` expresses the same branching as an optimisable expression.

In [43]:
if SPARK_AVAILABLE:
    with_udf = orders.withColumn("tier_udf", tier_udf(F.col("amount")))
    with_builtin = orders.withColumn(
        "tier_builtin",
        F.when(F.col("amount") >= 100, "high")
        .when(F.col("amount") >= 50, "mid")
        .otherwise("low"),
    )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Same answer, different cost

**Expected output (with Spark):**

```
UDF 'high' rows   : 2
Built-in 'high' rows: 2
```

In [44]:
if SPARK_AVAILABLE:
    with_udf.select("amount", "tier_udf").show()
    print("UDF 'high' rows   :", with_udf.filter(F.col("tier_udf") == "high").count())
    print("Built-in 'high' rows:", with_builtin.filter(F.col("tier_builtin") == "high").count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Interview note

The expected answer is "UDFs are a code smell in hot paths". Add the escape hatch: pandas UDFs
for vectorisation and, on Databricks, a Scala/Java UDF or `transform`/higher-order SQL
functions for zero-copy JVM execution.


### Expected output (with Spark)

```
self-check 8 passed: UDF and built-in agree
```

In [45]:
if SPARK_AVAILABLE:
    # --- self-check 8 ---------------------------------------------------
    # Same answer, but the built-in version can be optimised and code-generated.
    assert with_udf.filter(F.col("tier_udf") == "high").count() == 2
    assert with_builtin.filter(F.col("tier_builtin") == "high").count() == 2
    print("self-check 8 passed: UDF and built-in agree")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 10. Arrays and `explode`

### Concept

Semi-structured values (JSON lists, multi-valued tags) are natural as `array<T>` columns.

### The core operators

- `explode(col)` - one output row per element, **dropping null/empty arrays**.
- `explode_outer(col)` - same, but keeps a row with `null` when the array is null/empty.
- `array(...)`, `array_contains`, `array_distinct`, `size`, `concat`, `collect_list`
  for building and querying arrays;
- `posexplode` when the index matters.

`collect_list`/`collect_set` in a `groupBy` are the reverse operation: rows back into an array.

### Under the hood: explode is a row multiplier

`explode` is a generator: for each input row it emits one output row per element of the
array. Internally it uses a generate operator, and it can share the same stage as the
surrounding narrow operations, so it does not add a shuffle by itself. What it does add is
rows. A table with 10 million rows and an average array length of 20 becomes 200 million
rows after the explode, which enlarges every downstream aggregate, join, and file.

The two variants differ only in the empty case. `explode` drops a row whose array is null
or empty, because there is no element to emit. `explode_outer` keeps it once with a null
element. `posexplode` adds the element index, and `posexplode_outer` is the outer version
of that. When you explode and then `groupBy`, the `collect_list` / `collect_set`
aggregates are the inverse operation and rebuild arrays - but they do not preserve order
unless you sort first.

### Common mistake: losing rows to an empty array

The silent correctness bug is using `explode` on a column that can be null or empty and
then reporting a total. The rows whose arrays were empty do not vanish from the input -
they vanish from the output, and nothing raises. A count that should include them comes
back low.

Test the boundary explicitly: feed in a null array and an empty array, explode with both
variants, and assert the counts. If the element is genuinely optional, use
`explode_outer`; if an empty array means "no value", keep `explode` and document the
choice. The same care applies before exploding into a join: a hot array element creates
the same skew a hot key would.

### Expected output (with Spark)

```
explode rows      : 3   (u1 has 2 tags, u2 has 1, empty/null arrays vanish)
explode_outer rows: 5   (u3 and u4 appear with a null tag)
tables with tag 'new': 4
```

### Interview note

`explode` is a common hidden row multiplier - it inflates row counts and therefore file sizes
and shuffle volume. Mention the `explode_outer` choice explicitly: silently dropping rows with
empty arrays is a classic correctness bug in event pipelines.

### Build the input

`u3` has an empty array and `u4` has `null`.

**Expected output (with Spark):** the four-row frame with a `tags` array column.

In [46]:
if SPARK_AVAILABLE:
    users = spark.createDataFrame(
        [
            ("u1", ["spark", "sql"]),
            ("u2", ["python"]),
            ("u3", []),
            ("u4", None),
        ],
        "user_id string, tags array<string>",
    )
    users.show(truncate=False)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### `explode` vs `explode_outer`

**Expected output (with Spark):**

```
explode rows      : 3   (u1 has 2 tags, u2 has 1, empty/null arrays vanish)
explode_outer rows: 5   (u3 and u4 appear with a null tag)
```

In [47]:
if SPARK_AVAILABLE:
    exploded = users.select("user_id", F.explode("tags").alias("tag"))
    exploded_outer = users.select("user_id", F.explode_outer("tags").alias("tag"))
    print("explode rows      :", exploded.count(), "  (u1 has 2 tags, u2 has 1, empty/null arrays vanish)")
    print("explode_outer rows:", exploded_outer.count(), "  (u3 and u4 appear with a null tag)")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Building arrays with `concat` / `array_distinct`

`coalesce(tags, array())` replaces a null array with an empty one so `concat` always works.

**Expected output (with Spark):**

```
tables with tag 'new': 4
```

In [48]:
if SPARK_AVAILABLE:
    augmented = users.withColumn(
        "tags",
        F.array_distinct(
            F.concat(F.coalesce(F.col("tags"), F.array().cast("array<string>")), F.array(F.lit("new")))
        ),
    )
    augmented.show(truncate=False)
    print("tables with tag 'new':", augmented.filter(F.array_contains(F.col("tags"), "new")).count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 9 passed: explode row counts and array ops correct
```

In [49]:
if SPARK_AVAILABLE:
    # --- self-check 9 ---------------------------------------------------
    assert exploded.count() == 3
    assert exploded_outer.count() == 5
    assert augmented.filter(F.array_contains(F.col("tags"), "new")).count() == 4
    print("self-check 9 passed: explode row counts and array ops correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## Part 3 - Run it like production

The transformations in Parts 1-2 are half the job; what actually breaks schedules is
layout, memory and skew. This part walks the operational layer: file layout, caching,
reading plans, AQE, small files, streaming, table formats and which platform to name.


## 11. Partitioning and Parquet write/read with partition pruning

### Concept

`partitionBy` on a **low-cardinality** column writes the data into a directory per value
(`order_date=2026-10-01/part-....parquet`) and removes that column from the files themselves.

A later query that filters on the partition column can then read only the matching directories -
**partition pruning** - which is the single biggest I/O win in most lakehouse jobs.

### Parquet's extra wins

Parquet adds column pruning and predicate pushdown inside each file via its footer metadata,
plus compression and encoding.

### Choosing partition keys

Thousands of tiny `date=.../hour=.../user_id=...` directories is "over-partitioning" and is
worse than not partitioning at all. Target roughly 100 MB-1 GB per partition file.

### Under the hood: how pruning actually works

When you `write.partitionBy("order_date")`, Spark creates a directory per distinct value
and stores the partition column in the directory name, not inside the files. On read, the
catalog reconstructs `order_date` as a virtual column and the planner compares the filter
value against the directory layout. If the predicate matches only some directories, the
scan lists and reads only those - **partition pruning** - and the plan shows them in
`PartitionFilters`.

Inside each file, Parquet has its own pruners. The footer stores per-column min/max
statistics for every row group, so a filter can skip row groups whose ranges cannot match.
That is data skipping, and it works on non-partition columns as long as the predicate is a
simple comparison the statistics can answer. A function wrapped around the column
(`to_date(ts) = ...`, `upper(city) = ...`) hides the raw value and disables it.

The layout that wins is boring: partition on a low-to-medium cardinality column you almost
always filter on, keep files around 100 MB to 1 GB, and let row-group statistics handle
the rest.

### Common mistake: over-partitioning and filtering on a calculated column

Two failure modes dominate. The first is over-partitioning: `partitionBy("user_id")` on a
table with a million users creates a million directories, each holding a few kilobytes.
Listings become slow, the driver holds a huge directory tree, and the small-file problem
gets worse, not better. Partition by `date`, `region`, or `tenant`, not by a unique key.

The second is a predicate that cannot be pushed down. `filter(to_date(col("order_ts")) ==
lit("2026-10-01"))` computes a new value per row, so neither the partition layout (unless
you partition on the derived column) nor Parquet statistics can prune anything; the engine
reads everything and filters afterwards. Filter on the raw stored column, or persist the
derived column and partition on it.

### Expected output (with Spark)

```
root
 |-- order_id: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- order_date: date (nullable = true)

rows read back        : 6
rows for 2026-10-01   : 2
partition directories : ['order_date=2026-10-01', 'order_date=2026-10-02', 'order_date=2026-10-03']
```

### Interview note

Mention pruning **and** its limits: filtering on a non-partition column still scans every file
(it only skips row groups via pushdown). This is exactly why low-cardinality partition keys plus
Parquet stats are the standard lakehouse layout, and why tools like Delta/Iceberg add file-level
min/max statistics.

### Write a partitioned dataset

The directory is a temporary one, cleaned up at the end of the section.

In [50]:
if SPARK_AVAILABLE:
    import os
    import tempfile

    base_dir = tempfile.mkdtemp(prefix="de-interview-parquet-")
    parquet_path = os.path.join(base_dir, "orders_parquet")

    (
        orders
        .withColumn("order_date", F.to_date("order_ts"))
        .write
        .mode("overwrite")
        .partitionBy("order_date")
        .parquet(parquet_path)
    )
    print("wrote partitioned parquet to", parquet_path)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Read it back

**Expected output (with Spark):** the schema shown above, then `rows read back : 6`.

In [51]:
if SPARK_AVAILABLE:
    read_back = spark.read.parquet(parquet_path)
    read_back.printSchema()
    print("rows read back        :", read_back.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Partition pruning in the plan

The `explain()` output should list only the matching partition directory in `PartitionFilters`.

**Expected output (with Spark):** `rows for 2026-10-01   : 2`.

In [52]:
if SPARK_AVAILABLE:
    pruned = read_back.filter(F.col("order_date") == F.to_date(F.lit("2026-10-01")))
    print("rows for 2026-10-01   :", pruned.count())
    pruned.explain()  # the scan should list only the matching partition directory
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### The directories on disk

**Expected output (with Spark):** three `order_date=...` directories.

In [53]:
if SPARK_AVAILABLE:
    import glob

    partition_dirs = sorted(
        os.path.basename(p) for p in glob.glob(os.path.join(parquet_path, "order_date=*"))
    )
    print("partition directories :", partition_dirs)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 10 passed: partitioned Parquet round-trip with pruning
```

In [54]:
if SPARK_AVAILABLE:
    # --- self-check 10 --------------------------------------------------
    assert read_back.count() == 6
    assert pruned.count() == 2
    assert partition_dirs == [
        "order_date=2026-10-01",
        "order_date=2026-10-02",
        "order_date=2026-10-03",
    ]
    print("self-check 10 passed: partitioned Parquet round-trip with pruning")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Clean up the temporary directory

Nothing to assert here - it just keeps the workspace tidy.

In [55]:
if SPARK_AVAILABLE:
    shutil.rmtree(base_dir, ignore_errors=True)
    print("removed", base_dir)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 12. Caching / persist and when not to cache

### Concept

`cache()` = `persist(StorageLevel.MEMORY_AND_DISK)`. It stores the computed partitions so the
**next** action reuses them instead of recomputing the whole lineage.

`MEMORY_ONLY` drops partitions it cannot hold and recomputes them; `MEMORY_AND_DISK` spills to
disk; `DISK_ONLY` trades speed for memory. Always `unpersist()` when done - cached blocks
occupy executor memory that shuffle and joins also need.

### When NOT to cache

- the DataFrame is used once (cache is pure overhead);
- the source is already small/cheap, or you are going to `write` it immediately anyway;
- you cached "just in case" and the job now has memory pressure (the classic OOM culprit in long
  notebooks).

Also remember caching is **lazy** - the first action materialises it. Iterative algorithms (ML
training loops) and DataFrames reused 3+ times are the real wins.

### Under the hood: what cache stores and how it is evicted

`cache()` is `persist(MEMORY_AND_DISK)`. It marks the DataFrame as cached; the first
action computes each partition and stores the resulting blocks. The second action reads
those blocks instead of replaying the lineage.

Storage lives in the executor's unified memory region, which it shares with execution
memory (shuffle, joins, sorts). When cached blocks do not fit, Spark evicts the
least-recently-used partitions; with `MEMORY_ONLY` an evicted partition is simply
recomputed on demand, while `MEMORY_AND_DISK` spills it to local disk. `DISK_ONLY` and
`OFF_HEAP` trade speed and garbage-collection pressure for capacity. Cached blocks are
per-executor, so losing an executor loses its blocks even though the driver still thinks
the frame is cached. `unpersist()` frees the blocks and is not optional in a long-running
session.

### Common mistake: caching "just in case"

Caching is an optimization only when the frame is reused and expensive. The common
anti-patterns:

- Caching a frame used by exactly one action: the write to the cache store is pure
  overhead, and the job can end up slower than recomputing.
- Caching a trivial filter over a small table: recomputation is nearly free, and the cache
  only adds memory pressure.
- Caching and forgetting `unpersist()`: the blocks occupy unified memory that the next
  shuffle or join needs, a classic cause of spill and executor OOM.
- Caching to "fix" a slow job: if the bottleneck is one skewed partition or a missing
  filter, the cache hides the symptom for one action and changes nothing structurally.

The rule of thumb: cache an expensive intermediate reused by three or more actions, or a
frame inside an iterative loop, and measure the job with and without it.

### Expected output (with Spark)

```
is_cached after cache()      : True
rows (materialises cache)    : 3
rows from cache              : 3
is_cached after unpersist()  : False
persisted with MEMORY_AND_DISK: True
```

### Interview note

Contrast `cache` with `checkpoint`. Cache keeps the **lineage** and is lost when the session
dies; checkpoint truncates the lineage to a reliable store, which is what you need for very long
chains or exactly-once recovery.

### Cache an expensive aggregate

`is_cached` is `True` immediately after `cache()` (the plan is marked), but no data is stored
until the first action.

In [56]:
if SPARK_AVAILABLE:
    from pyspark import StorageLevel

    expensive = (
        orders
        .withColumn("order_date", F.to_date("order_ts"))
        .groupBy("order_date")
        .agg(F.sum("amount").alias("revenue"))
        .cache()
    )
    print("is_cached after cache()      :", expensive.is_cached)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Materialise, then reuse

**Expected output (with Spark):**

```
rows (materialises cache)    : 3
rows from cache              : 3
```

In [57]:
if SPARK_AVAILABLE:
    first = expensive.count()   # first action materialises the cached blocks
    second = expensive.count()  # served from memory/disk, no recompute
    print("rows (materialises cache)    :", first)
    print("rows from cache              :", second)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Unpersist and use an explicit storage level

**Expected output (with Spark):**

```
is_cached after unpersist()  : False
persisted with MEMORY_AND_DISK: True
```

In [58]:
if SPARK_AVAILABLE:
    expensive.unpersist()
    print("is_cached after unpersist()  :", expensive.is_cached)

    persisted = orders.persist(StorageLevel.MEMORY_AND_DISK)
    persisted.count()
    print("persisted with MEMORY_AND_DISK:", persisted.is_cached)
    persisted.unpersist()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 11 passed: cache lifecycle correct
```

In [59]:
if SPARK_AVAILABLE:
    # --- self-check 11 --------------------------------------------------
    assert first == 3 and second == 3
    assert not expensive.is_cached
    assert not persisted.is_cached
    print("self-check 11 passed: cache lifecycle correct")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 13. Reading `explain()` plans

### Concept

`df.explain()` prints the physical plan; `explain(True)` also prints parsed, analysed and
optimised logical plans. Read it bottom-up (leaves to root).

### What to look for

- **Scan** - `FileScan parquet ... PushedFilters: [...]`; what was read and which predicates
  were pushed down. `PartitionFilters` shows partition pruning.
- **Filter / Project** - narrow work that stayed in-partition.
- **Exchange** - **a shuffle**. Any `Exchange` in the plan is your cost indicator.
- **HashAggregate** - a partial aggregate before the exchange, then a final one after.
- **BroadcastHashJoin** vs **SortMergeJoin** - whether the small side was broadcast.
- **AdaptiveSparkPlan** - AQE is enabled and may re-optimise at runtime.

### `EXPLAIN FORMATTED` as data

`spark.sql("EXPLAIN FORMATTED <query>")` returns the plan as a DataFrame, which is handy for
asserting on the plan in tests.

### Under the hood: the four plan stages

`explain(True)` shows four plans. The **parsed** plan is the literal syntax. The
**analysed** plan resolves names and types against the catalog and expands `*`. The
**optimised logical** plan is where Catalyst rules fire: predicate pushdown, column
pruning, constant folding, join reordering, and converting a filter over a join into a
filter on one side. The **physical** plan is the operators that actually run, already
chosen by the planner.

Read it bottom-up. Leaves are scans; parents are operations on their output. The nodes
that consume time are the ones that move or store data: `Exchange` (shuffle), `Sort`,
`HashAggregate` (especially with spill), and joins. `WholeStageCodegen` wraps a group of
operators that were compiled into one generated function, which is a sign the fast path
was preserved. `AdaptiveSparkPlan isFinalPlan=false` means the printed plan may still
change at runtime.

### Common mistake: reading the plan top-down and trusting the estimate

The plan tree prints the final result at the top and the sources at the bottom, so reading
it as prose inverts the data flow. Start at the leaves: which files and partitions were
scanned, how many columns, and which filters were pushed.

The other trap is treating `explain()` as a performance measurement. It shows the plan
Catalyst chose at planning time, using estimated statistics that can be wrong by orders of
magnitude, and it does not show actual row counts, spill, or skew. A plan with no
`Exchange` can still be slow if one task reads a huge partition. Use the plan to confirm
shape (pruning, join strategy, pushdown), then use the Spark UI or the query metrics to
confirm cost.

### Expected output (with Spark)

```
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[status#... ], functions=[sum(amount#...)])
   +- Exchange hashpartitioning(status#..., 8)
      +- HashAggregate(keys=[status#...], functions=[partial_sum(amount#...)])
         +- FileScan parquet ...
```

### Interview note

Being able to point at the `Exchange` and say "that is the shuffle, and here is why" is the
strongest practical signal in a Spark interview. Bring a real example: a `groupBy` on a
high-cardinality key with two `Exchange` nodes and a post-shuffle skew.

### Get the plan for a `GROUP BY`

**Expected output (with Spark):** the formatted plan, including an `Exchange`.

In [60]:
if SPARK_AVAILABLE:
    orders.createOrReplaceTempView("orders")

    plan_rows = spark.sql(
        """
        EXPLAIN FORMATTED
        SELECT status, sum(amount) AS revenue
        FROM orders
        GROUP BY status
        """
    ).collect()
    plan_text = "\n".join(row[0] for row in plan_rows)
    print(plan_text)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Assert on the plan text

A `GROUP BY` is wide, so the plan must contain an `Exchange`. A test that asserts this is a
cheap regression guard for accidentally breaking predicate/pipeline behaviour.

**Expected output (with Spark):**

```
plan contains an Exchange (shuffle) as expected for GROUP BY
```

In [61]:
if SPARK_AVAILABLE:
    lowered = plan_text.lower()
    assert len(plan_text) > 0
    assert "aggregate" in lowered
    # The GROUP BY is a wide transformation, so a shuffle must appear in the plan.
    assert "exchange" in lowered, plan_text
    print("plan contains an Exchange (shuffle) as expected for GROUP BY")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 14. Adaptive Query Execution (AQE) and skew handling

### Concept

Since Spark 3.0, AQE re-optimises the plan **at runtime** using statistics from completed
shuffle stages.

### The three big features

1. **Coalesce post-shuffle partitions** - `spark.sql.adaptive.coalescePartitions.enabled=true`
   merges many small shuffle partitions into `advisoryPartitionSizeInBytes`-sized ones. This is
   the modern answer to "my job has 200 tiny tasks".
2. **Skew join handling** - `spark.sql.adaptive.skewJoin.enabled=true` detects
   disproportionately large partitions and splits them into sub-tasks (usually by salting
   internally), so one straggler task no longer defines the job duration.
3. **Convert sort-merge join to broadcast join** at runtime if a side turns out to be small.

### Manual skew handling

Without AQE, skew is handled manually: **salt** the hot keys (append a random suffix, explode
the other side), then re-aggregate.

### Under the hood: when AQE re-plans

Without AQE, the plan is final before the first task runs, so a wrong estimate - an
implicit broadcast that is too large, or 200 shuffle partitions for 1 GB of data - is paid
in full. With `spark.sql.adaptive.enabled=true`, Spark inserts an `AdaptiveSparkPlan` at
each shuffle boundary. It finishes the map side, reads the shuffle statistics (the actual
compressed sizes per partition), and re-optimises the reduce side before scheduling it.

That is what enables the three features. Coalescing merges adjacent small shuffle
partitions into ones near `advisoryPartitionSizeInBytes` (default 64 MB), so the task
count follows the data rather than the configured number. Skew join detection splits an
oversized partition into sub-partitions and joins each against the matching side, which is
internal salting. The join strategy can also be switched to broadcast at runtime when a
side turns out small.

### Common mistake: expecting AQE to fix a skewed read

AQE re-optimises **after** a shuffle, using the statistics that shuffle produced. If the
skew already exists in how the data is laid out - one Parquet file much larger than the
rest, one Kafka partition carrying most of the traffic, one key dominating the map-side
output - the first stage is still the straggler, and AQE cannot split an input file or a
source partition.

The second misconception is that AQE replaces tuning. It cannot fix a missing filter, a
Python UDF on the hot path, or a `collect()` that pulls a billion rows to the driver. Turn
it on, then read the plan and the UI as usual. Also check version behaviour: AQE is
enabled by default from Spark 3.2, but `skewJoin.enabled` and coalescing are separate
flags, and AQE support in Structured Streaming has been limited, so streaming skew still
needs manual design.

### Watch AQE change the partition count

The code below makes 2,000 tiny input partitions, groups them, and prints the number of
partitions on the reduce side. With AQE coalescing on, that number is much smaller than
the configured shuffle count, because Spark measured the shuffle output and packed it.

In [62]:
if SPARK_AVAILABLE:
    # AQE coalescing made concrete: many small post-shuffle partitions.
    tiny = spark.range(0, 2000, 1, 2000)
    print("input partitions          :", tiny.rdd.getNumPartitions())

    grouped = tiny.groupBy((F.col("id") % 4).alias("bucket")).count()
    print("post-shuffle partitions   :", grouped.rdd.getNumPartitions())
    print("rows per bucket           :", sorted(row["count"] for row in grouped.collect()))
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")

SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
AQE enabled               : true
coalescePartitions        : true
skewJoin enabled          : true
advisory partition size   : 64m
+---------+---+
|status   |n  |
+---------+---+
|paid     |3  |
|cancelled|1  |
|returned |1  |
|shipped  |1  |
+---------+---+
```

### Turn the knobs

**Expected output (with Spark):** the four configuration values echoed back.


In [63]:
if SPARK_AVAILABLE:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")

    print("AQE enabled               :", spark.conf.get("spark.sql.adaptive.enabled"))
    print("coalescePartitions        :", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
    print("skewJoin enabled          :", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
    print("advisory partition size   :", spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Probe the skew

`explain()` shows an `AdaptiveSparkPlan`: the plan is re-planned after shuffle statistics
arrive.

In [64]:
if SPARK_AVAILABLE:
    skew_probe = spark.sql(
        "SELECT status, count(*) AS n FROM orders GROUP BY status ORDER BY n DESC, status"
    )
    skew_probe.show()
    skew_probe.explain()
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 12 passed: AQE flags set and skew probe returned every status
```

In [65]:
if SPARK_AVAILABLE:
    # --- self-check 12 --------------------------------------------------
    assert spark.conf.get("spark.sql.adaptive.enabled") == "true"
    assert spark.conf.get("spark.sql.adaptive.skewJoin.enabled") == "true"
    assert skew_probe.count() == 4
    print("self-check 12 passed: AQE flags set and skew probe returned every status")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Interview note

Say what AQE *cannot* fix - a single key holding most rows still ends up in one partition; AQE
splits it, but if the source read itself is skewed you may need to rebalance the input
(`repartition` by a salted key, or fix upstream). Also note AQE is disabled for Structured
Streaming in many versions, so streaming skew needs manual design.


## 15. Small files and compaction

### Concept

Each Spark task writes one file per partition. A job with 2,000 partitions writes 2,000 files;
streaming or hourly ingestion multiplies that.

### Why small files hurt

- every file has footer/metadata overhead in the driver and in the scan plan;
- object stores (S3) list and open files with high per-request latency;
- query planning times out on millions of files.

### Fixes

`coalesce(n)` (narrow, no shuffle - it just merges partitions, best when reducing slightly) or
`repartition(n)` (full shuffle, even sizes - best when the current sizes are wildly uneven).
Then rewrite the table.

Dedicated compaction jobs (Databricks `OPTIMIZE`, Iceberg `rewrite_data_files`, Hudi
compaction) do the same thing periodically.

### Under the hood: one task writes one file

Output file count is driven by the number of tasks in the final write stage, because each
task writes its own file (plus one per partition directory if you use `partitionBy`). So a
job with 2,000 tasks over ten partition directories can write up to 20,000 files.
Streaming and hourly batch jobs multiply that by the number of micro-batches or hours.

Small files are expensive on both ends. Writing costs one object-store PUT and one file
footer each. Reading costs one listing entry and one open per file; if
`spark.sql.files.maxPartitionBytes` (default 128 MB) cannot group them, Spark schedules one
task per file, and the scheduler spends more time on task overhead than on data. Planning
also degrades: the driver holds the file list and computes splits for every one of them.

### Common mistake: coalesce(1) as a compaction strategy

`coalesce(1)` removes the shuffle and writes a single file, which looks tidy on a small
dataset and is a disaster on a large one: one task reads and writes all the data, so the
job loses parallelism and can OOM or time out. `coalesce(n)` is also only a merge of
existing partitions, so if the input partitions are wildly uneven the output files stay
uneven. Use `repartition(n)` for even sizes when the current distribution is bad.

The right production answer is a scheduled compaction job - Delta `OPTIMIZE`, Iceberg
`rewrite_data_files`, Hudi compaction - or a periodic rewrite with a bounded target file
size, not ad-hoc `coalesce(1)` scattered through every consumer. And compact before the
file count makes the listing itself the bottleneck.

### Expected output (with Spark, illustrative)

```
small files before compaction: 24
files after compaction       : 2
rows preserved               : 6
```

The exact file counts depend on `repartition(8)` output and Spark's task count, but the point
is stable: fewer, larger files with identical rows.

### Interview note

Mention the **read** side too - `spark.sql.files.maxPartitionBytes` controls how much a single
task reads; too many small files means one task per file and a storm of tiny tasks. The usual
production answer is a scheduled compaction job, not ad-hoc `coalesce` in every consumer.

### Write deliberately fragmented files

`repartition(8)` inflates the task count so each day writes up to eight tiny files.

In [66]:
if SPARK_AVAILABLE:
    small_base = tempfile.mkdtemp(prefix="de-interview-smallfiles-")
    raw_dir = os.path.join(small_base, "raw")
    compact_path = os.path.join(small_base, "compacted")

    for day in ("2026-10-01", "2026-10-02", "2026-10-03"):
        (
            orders
            .filter(F.to_date("order_ts") == F.to_date(F.lit(day)))
            .repartition(8)  # deliberately inflate the task -> file count
            .write
            .mode("overwrite")
            .parquet(os.path.join(raw_dir, f"ds={day}"))
        )
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Count the files before compaction

**Expected output (with Spark, illustrative):** `small files before compaction: 24`.

In [67]:
if SPARK_AVAILABLE:
    before = [
        os.path.join(root, f)
        for root, _, files in os.walk(raw_dir)
        for f in files
        if f.endswith(".parquet")
    ]
    print("small files before compaction:", len(before))
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Compact with `coalesce`

The compacted copy goes to a sibling directory so re-reading `raw_dir` after the write cannot
pick up the compacted files (which would double-count the rows).

In [68]:
if SPARK_AVAILABLE:
    raw = spark.read.option("recursiveFileLookup", "true").parquet(raw_dir)
    compacted = raw.coalesce(2)  # merge partitions without a shuffle
    compacted.write.mode("overwrite").parquet(compact_path)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Count the files after compaction

**Expected output (with Spark, illustrative):**

```
files after compaction       : 2
rows preserved               : 6
```

In [69]:
if SPARK_AVAILABLE:
    after = [
        os.path.join(root, f)
        for root, _, files in os.walk(compact_path)
        for f in files
        if f.endswith(".parquet")
    ]
    print("files after compaction       :", len(after))
    print("rows preserved               :", compacted.count())
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


### Expected output (with Spark)

```
self-check 13 passed: compaction reduced file count and preserved rows
```

In [70]:
if SPARK_AVAILABLE:
    # --- self-check 13 --------------------------------------------------
    assert compacted.count() == 6
    assert 0 < len(after) < len(before)
    print("self-check 13 passed: compaction reduced file count and preserved rows")
    shutil.rmtree(small_base, ignore_errors=True)
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell


## 16. Structured Streaming sketch (code shown, never executed)

Structured Streaming treats a stream as an unbounded table you keep querying. Three things an
interviewer listens for:

- **`readStream` / `writeStream`** - the same DataFrame API as batch, so joins and aggregations
  transfer directly.
- **Checkpointing** - `checkpointLocation` persists offsets and state so a restart resumes where
  it stopped instead of reprocessing or losing data.
- **Output mode / watermark / trigger** - `append` vs `update` vs `complete`, and a watermark to
  bound state for event-time aggregations.

The block below is intentionally **not executed** by this notebook (it needs a Kafka broker and
a table sink). Read it as a reference shape.

### Under the hood: micro-batches and the state store

Structured Streaming runs the batch DataFrame engine in a loop. Each trigger fires a
micro-batch: read the next offsets from the source, plan and execute the same query as a
batch job, commit the result to the sink, then record the new offsets. `readStream`
produces an unbounded logical table, and every batch is the delta since the last commit.

Stateful operations (`groupBy` with a window, streaming joins, dedupe) keep their
intermediate rows in a state store on the executors, checkpointed to the durable
`checkpointLocation`. That checkpoint holds the committed offsets and the state, which is
what lets a restart resume rather than reprocess. Without a watermark, event-time
aggregations must keep state for every key forever, so the store grows until the job
fails.

### Common mistake: update mode without a watermark, and the wrong sink

`outputMode("update")` emits only the rows that changed in a batch, which is attractive
for dashboards. But if the aggregation has no watermark, its state is unbounded: every
distinct key stays in the state store forever, so the job's memory grows with the key
space rather than with time. Add a watermark to bound how late data can arrive, and use
`append` when you only need finalized windows.

The second mistake is assuming the sink makes the job exactly-once. Exactly-once needs a
replayable source, a durable checkpoint, and an idempotent or transactional sink. A plain
file sink or a non-idempotent `foreachBatch` gives at-least-once on retry. `foreachBatch`
can be made idempotent by keying the write on the batch id, but that is your
responsibility, not Spark's.

```python
# Reference only - NOT executed by scripts/run_notebooks.py.
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, TimestampType

order_schema = StructType() \
    .add("order_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("amount", DoubleType()) \
    .add("status", StringType()) \
    .add("order_ts", TimestampType())

stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "latest")
    .load()
)
```

```python
parsed = (
    stream
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .withWatermark("order_ts", "10 minutes")
)

query = (
    parsed
    .groupBy(F.window("order_ts", "5 minutes"), "status")
    .agg(F.count("*").alias("n"), F.sum("amount").alias("revenue"))
    .writeStream
    .outputMode("update")
    .option("checkpointLocation", "s3://company-checkpoints/orders_5m/")
    .trigger(processingTime="1 minute")
    .toTable("main.gold.orders_5m")  # or .foreachBatch(...) / .format("delta").start(path)
)

query.awaitTermination()
```

**Interview note:** the killer follow-up is **exactly-once**. Structured Streaming gives
end-to-end exactly-once only when the source is replayable (Kafka with committed offsets), the
checkpoint is durable, and the sink is idempotent/transactional (Delta, Iceberg, or a
`foreachBatch` merge keyed on a deterministic batch id). `update` mode without a watermark means
unbounded state.

## 17. Delta Lake and Apache Iceberg notes (mostly a Databricks conversation)

Both projects add a **transaction log / metadata layer** on top of Parquet, which is what turns
a directory of files into a reliable table. What they share:

- **ACID transactions and atomic metadata commits** - a failed write does not leave a
  half-visible table.
- **Time travel and snapshots** - query `VERSION AS OF` / `TIMESTAMP AS OF`.
- **Schema evolution and enforcement** - `mergeSchema`, `ALTER TABLE ... ADD COLUMN`.
- **`MERGE` / upsert** - the standard CDC pattern: match on the primary key, update matched
  rows, insert new ones, delete flagged ones.
- **Small-file management** - Delta `OPTIMIZE` (with `ZORDER BY` for data skipping), Iceberg
  `rewrite_data_files` + `rewrite_manifests`.
- **Statistics for data skipping** - per-file min/max, so `WHERE` on non-partition columns can
  still prune files.

### Under the hood: how a table format works

A directory of Parquet files has no notion of a transaction, so a reader can see a
half-written dataset and a writer cannot atomically commit several files. Delta Lake and
Iceberg fix this with a metadata layer. Iceberg keeps a metadata JSON tree plus manifest
files listing data files and their statistics; Delta keeps a `_delta_log` of JSON commits
and checkpoint Parquet files. Both write a new metadata entry as the commit point, and
readers resolve the current snapshot from it.

That design is what produces the features: time travel (old metadata entries are still
readable until they are expired), ACID writes (a commit is a single atomic metadata swap),
schema evolution (the metadata records the schema and its history), and data skipping
(manifest or log entries carry per-file min/max so a query can skip files without listing
them). Compaction rewrites data files and records the replacement in a new commit.

### Common mistake: treating the table as just Parquet

The most common operational mistake is reading or writing the underlying files directly,
or pointing a second engine at the directory while another engine owns the log. You lose
the snapshot guarantee, can read uncommitted files, and can corrupt the table history.
Query the table through an engine that understands the format.

The other mistakes are maintenance-related. Time travel retains old files, so snapshots
must be expired (`VACUUM` for Delta, `expire_snapshots` for Iceberg) or storage grows
forever; but vacuum too aggressively and time travel stops working, so align the retention
window with the log retention. Small files accumulate after every `MERGE`, so compaction is
scheduled work, not an optional cleanup. And `MERGE` on a non-unique source key can produce
duplicate matches, which the format will not prevent for you.

### Differences worth calling out

| | Delta Lake | Apache Iceberg |
|---|---|---|
| Metadata | JSON + Parquet transaction log (`_delta_log`) | Metadata JSON + manifest files |
| Format spec | Delta protocol, originally Databricks | Open table spec, vendor-neutral |
| Hidden partitioning | Partition columns are real columns | `PARTITION BY` transforms (`days(ts)`, `bucket(id, 16)`) hide the column |
| Pushdown | Data skipping + ZORDER | Partition transforms + sort orders + manifests |
| Typical home | Databricks, Spark, some Trino/Flink | Trino, Flink, Spark, Snowflake, BigQuery |

### Databricks-specific habits

Unity Catalog for governance, `OPTIMIZE` + `ZORDER`, `VACUUM` to expire old files, `MERGE INTO`
for upserts, and `AUTO OPTIMIZE` / `AUTO COMPACT` for streaming sinks. On Databricks the
**Photon** engine and **predictive optimization** do much of the tuning, but partitioning, skew,
and file sizing are still design decisions you own.

**Expected output:** markdown only - nothing to execute.

**Interview note:** if asked "Delta or Iceberg?", answer with constraints: what query engines
must read the table, whether you need vendor neutrality, whether hidden partitioning and sort
orders matter, and what the team already runs. Then talk about the cost of migration - metadata
is the easy part; downstream `MERGE` jobs and permission models are not.

## 18. Platform translation: local PySpark, Databricks and Snowflake

### Concept

Most interview Spark examples run locally, but production answers should name the platform boundary. The same DataFrame transformation can run as plain PySpark on Kubernetes, as a Databricks job against Delta tables, or be replaced by Snowflake SQL when the data already lives in Snowflake and the operation is relational.

### How to choose the engine

Use PySpark or Databricks when the work is file-heavy, streaming, ML-adjacent, custom-code-heavy, or needs Delta Lake features close to object storage. Use Snowflake when the work is governed SQL over warehouse tables, when workload isolation with virtual warehouses is the cleaner operational model, or when business users already consume the curated marts there.

### Interview note

A strong answer avoids platform tribalism. Say where the data lives, what engine can prune the least data, which governance layer owns access, and which cost meter is easier to control.

### Decision matrix

| Workload clue | Better first answer | Why |
| --- | --- | --- |
| Kafka stream into bronze/silver with replay | Databricks / PySpark Structured Streaming | Checkpoints, state store and Delta commits are the natural controls. |
| Daily BI mart already in Snowflake | Snowflake SQL task or dynamic table | No data movement, warehouse isolation, RBAC and query history are already there. |
| Heavy joins over object-storage Parquet/Delta | Databricks / PySpark | File layout, shuffle tuning and Delta optimization are close to the data. |
| Ad hoc analyst dashboard with concurrency spikes | Snowflake virtual warehouse or Databricks SQL warehouse | Pick the platform where governance and serving tables already live. |
| Python library call per record | PySpark only after native SQL/functions fail | UDFs are expensive; prove that the custom code is necessary. |

### Cost model translation

Snowflake cost usually starts with warehouse credits and seconds running; Databricks cost starts with DBUs plus cloud VMs while a cluster or SQL warehouse is alive. In both cases, the first fix is rarely "make compute bigger forever". First remove columns, prune partitions/files, compact small files, isolate workloads, and shut idle compute down.

In [71]:
# Platform choice as a deterministic interview heuristic.
workloads = {
    "cdc_stream_to_bronze": {"streaming": True, "data_in_snowflake": False, "needs_python": False},
    "finance_mart_refresh": {"streaming": False, "data_in_snowflake": True, "needs_python": False},
    "feature_backfill": {"streaming": False, "data_in_snowflake": False, "needs_python": True},
}

def choose_platform(workload):
    if workload["streaming"] or workload["needs_python"]:
        return "Databricks/PySpark"
    if workload["data_in_snowflake"]:
        return "Snowflake SQL"
    return "Either: choose by governance, file layout and cost"

assert choose_platform(workloads["cdc_stream_to_bronze"]) == "Databricks/PySpark"
assert choose_platform(workloads["finance_mart_refresh"]) == "Snowflake SQL"
assert choose_platform(workloads["feature_backfill"]) == "Databricks/PySpark"
print({name: choose_platform(shape) for name, shape in workloads.items()})


{'cdc_stream_to_bronze': 'Databricks/PySpark', 'finance_mart_refresh': 'Snowflake SQL', 'feature_backfill': 'Databricks/PySpark'}


### Production checklist

- For Databricks: state whether the job runs on a job cluster or SQL warehouse, which Unity Catalog objects it reads/writes, how Delta files are optimized, and what cluster policy controls idle cost.
- For Snowflake: state the warehouse size, auto-suspend value, role, database/schema, expected pruning, and whether a task, stream, dynamic table or dbt job owns the refresh.
- For PySpark anywhere: state the input size, target file size, shuffle partitions, skew plan, checkpointing strategy for streaming, and how you will prove the job is idempotent.

## Part 4 - Sharpen the arithmetic without a cluster

No Spark needed here. These exercises train the mental arithmetic behind Parts 1-3, so
a shuffle size or a salt count never surprises you.


## PySpark exercises (standard library only)

These exercises need no Spark and no Java: they model the arithmetic and the failure modes
behind the concepts above with plain Python. Run each cell and compare the printed numbers
with the **Expected result** line.

The point is not the arithmetic itself. It is that the same quantities - partition sizes,
broadcast thresholds, per-partition counts, salt cardinality, shuffle bytes, window frame
bounds, null counts - are what you reason about when you tune a real job. If you can
compute them by hand, you can read a Spark UI or a plan and know which knob to turn.

Each exercise gives you a task, a hint, and the expected result. Try it before reading the
code.

### Try it 1. Size the shuffle partitions

**Task.** A `groupBy` shuffles 480 GB. Compute how many partitions you need so each
post-shuffle partition holds about 128 MB, then decide whether Spark's default of 200
shuffle partitions is enough.

**Hint.** Convert to a common unit first: `partitions = ceil(total_mb / target_mb)`. More
partitions than cores is fine; the scheduler runs them in waves.

**Expected result.** `3840 partitions; default 200 is too few`.

In [72]:
from math import ceil

total_gb = 480
target_mb = 128
default_shuffle_partitions = 200

total_mb = total_gb * 1024
needed = ceil(total_mb / target_mb)
verdict = "too few" if default_shuffle_partitions < needed else "enough"

assert needed == 3840, needed
print(f"{needed} partitions; default {default_shuffle_partitions} is {verdict}")

3840 partitions; default 200 is too few


### Try it 2. Broadcast or sort-merge?

**Task.** For each dimension table, decide whether Spark should broadcast it. The default
threshold is 10 MB, and a safe margin is half the threshold. Then pick the strategy for
each table.

**Hint.** Compare megabytes: `size_mb <= threshold_mb * margin` means broadcast. Anything
else falls back to a shuffle join.

**Expected result.** `catalog: broadcast; customers: broadcast; orders_history: sort-merge`.

In [73]:
threshold_mb = 10
margin = 0.5


def choose_strategy(size_mb, threshold_mb=threshold_mb, margin=margin):
    if size_mb <= threshold_mb * margin:
        return "broadcast"
    return "sort-merge"


tables = {"catalog": 2, "customers": 4, "orders_history": 350}
decisions = {name: choose_strategy(size) for name, size in tables.items()}

assert decisions == {
    "catalog": "broadcast",
    "customers": "broadcast",
    "orders_history": "sort-merge",
}, decisions
print("; ".join(f"{name}: {decision}" for name, decision in decisions.items()))

catalog: broadcast; customers: broadcast; orders_history: sort-merge


### Try it 3. Detect skew from per-partition counts

**Task.** Given the record counts of five shuffle partitions, compute the largest / median
ratio and flag skew when the ratio exceeds 3.

**Hint.** Take the median, then compare `max / median`. One giant partition among healthy
ones is the signature of a hot key.

**Expected result.** `max/median = 50.0, verdict = skewed`.

In [74]:
import statistics

partition_counts = [100, 101, 99, 100, 5000]
median = statistics.median(partition_counts)
ratio = max(partition_counts) / median
skewed = ratio > 3

assert median == 100
assert ratio == 50.0
assert skewed is True
print(f"max/median = {ratio:.1f}, verdict = {'skewed' if skewed else 'balanced'}")

max/median = 50.0, verdict = skewed

### Try it 4. Salt a hot key

**Task.** One customer key holds 1,000 of 1,800 rows. Splitting it across 8 salts spreads
that key's rows evenly. Compute the largest partition load before and after salting, and
the speedup on the hot key.

**Hint.** Before salting, the hot key's rows all land in one partition. After salting the
key becomes eight keys, so the hot-key load is `ceil(rows / salts)`.

**Expected result.** `largest partition before = 1000`, `after = 125`, speedup `8x`.

In [75]:
hot_key_rows = 1000
other_rows = 800
salts = 8

before = hot_key_rows  # every hot-key row lands in one partition
after = -(-hot_key_rows // salts)  # ceil division: 125 per salt bucket

assert other_rows + hot_key_rows == 1800
assert before == 1000
assert after == 125
print(f"largest partition before = {before}")
print(f"largest partition after  = {after} ({salts} salt buckets)")
print(f"hot-key speedup          = {before / after:.0f}x")

largest partition before = 1000
largest partition after  = 125 (8 salt buckets)
hot-key speedup          = 8x


### Try it 5. AQE coalescing math

**Task.** After a shuffle Spark has 200 partitions of about 5 MB each. With AQE
coalescing and an advisory size of 64 MB, how many partitions does the reduce side use?

**Hint.** AQE packs adjacent small partitions together and targets
`advisoryPartitionSizeInBytes`: `target = ceil(total_mb / advisory_mb)`, never more than
the starting count.

**Expected result.** `total = 1000 MB, advisory = 64 MB, coalesced partitions = 16`.

In [76]:
from math import ceil

post_shuffle_partitions = 200
partition_mb = 5
advisory_mb = 64

total_mb = post_shuffle_partitions * partition_mb
coalesced = min(post_shuffle_partitions, ceil(total_mb / advisory_mb))

assert total_mb == 1000
assert coalesced == 16
print(f"total = {total_mb} MB, advisory = {advisory_mb} MB, coalesced partitions = {coalesced}")

total = 1000 MB, advisory = 64 MB, coalesced partitions = 16


### Try it 6. Estimate shuffle bytes

**Task.** A `groupBy` reads 50 million rows of 180 bytes each. Estimate the shuffle write
(the map side writes one copy), the shuffle read (the reduce side reads it back), and the
total volume moved. Report in GiB.

**Hint.** Shuffle write is `rows * row_bytes`. Read is the same volume, so the total moved
is `2 * rows * row_bytes`. Divide by 1024 three times for GiB.

**Expected result.** about `8.38 GiB` written and `16.76 GiB` moved in total.

In [77]:
rows = 50_000_000
row_bytes = 180
gib = 1024 ** 3

written_bytes = rows * row_bytes
moved_bytes = 2 * written_bytes

assert written_bytes == 9_000_000_000
assert round(written_bytes / gib, 2) == 8.38
assert round(moved_bytes / gib, 2) == 16.76
print(f"shuffle write = {written_bytes / gib:.2f} GiB")
print(f"total moved   = {moved_bytes / gib:.2f} GiB (write + read)")

shuffle write = 8.38 GiB
total moved   = 16.76 GiB (write + read)


### Try it 7. Window frame arithmetic

**Task.** For the frame `rowsBetween(-2, 0)` (two rows before the current row through the
current row), compute the running sum at each position of `[10, 20, 30, 40, 50]`. The
first rows have a smaller frame because the window cannot start before the partition
begins.

**Hint.** For row `i`, the frame is `values[max(0, i - 2):i + 1]`. Sum that slice, and
count its length to see the frame size.

**Expected result.** running sums `[10, 30, 60, 90, 120]` and frame sizes `[1, 2, 3, 3, 3]`.

In [78]:
values = [10, 20, 30, 40, 50]
frame_size = 3  # -2 through the current row

running = [sum(values[max(0, i - (frame_size - 1)):i + 1]) for i in range(len(values))]
sizes = [len(values[max(0, i - (frame_size - 1)):i + 1]) for i in range(len(values))]

assert running == [10, 30, 60, 90, 120], running
assert sizes == [1, 2, 3, 3, 3], sizes
print("running sums:", running)
print("frame sizes :", sizes)

running sums: [10, 30, 60, 90, 120]
frame sizes : [1, 2, 3, 3, 3]


### Try it 8. Null versus empty semantics

**Task.** Given three rows with amounts `10.0`, `None`, `20.0`, compute `count(*)`,
`count(amount)`, and `avg(amount)` the way Spark does. Then check how `explode` treats an
empty list versus a null list: count the output rows for each.

**Hint.** `count(*)` counts rows, `count(col)` counts non-null values, and `avg` divides
the non-null sum by the non-null count. For explode, an empty list and a null list both
produce zero rows; `explode_outer` keeps one row with a null element.

**Expected result.** `count(*) = 3, count(amount) = 2, avg = 15.0`; explode drops both
empty and null, explode_outer keeps both.

In [79]:
rows = [{"amount": 10.0}, {"amount": None}, {"amount": 20.0}]

count_star = len(rows)
non_null = [row["amount"] for row in rows if row["amount"] is not None]
count_amount = len(non_null)
avg_amount = sum(non_null) / count_amount


def explode(arrays):
    return [element for array in arrays for element in (array or [])]


def explode_outer(arrays):
    out = []
    for array in arrays:
        if array:
            out.extend(array)
        else:
            out.append(None)
    return out


tag_arrays = [[], None]
assert (count_star, count_amount, avg_amount) == (3, 2, 15.0)
assert explode(tag_arrays) == []
assert explode_outer(tag_arrays) == [None, None]
print(f"count(*) = {count_star}, count(amount) = {count_amount}, avg = {avg_amount}")
print("explode on [[], None]       ->", explode(tag_arrays), "rows")
print("explode_outer on [[], None] ->", explode_outer(tag_arrays), "rows")

count(*) = 3, count(amount) = 2, avg = 15.0
explode on [[], None]       -> [] rows
explode_outer on [[], None] -> [None, None] rows


## Part 5 - Answer out loud

Answer each question aloud before reading the answer. The questions mirror Parts 1-3,
so a weak answer tells you exactly which section to re-run.


## Interview questions

Fifteen questions that cover the ground above. Answer each out loud before reading the answer.

**1. What is a shuffle, and why is it expensive?**

A shuffle redistributes rows across partitions so that rows sharing a key (or a partition
requirement) land together. It is expensive because it writes intermediate data to executor
local disk, transfers it across the network, reads it back, and can spill. It also forces a
stage boundary, so the whole upstream stage must complete first. Wide transformations cause
shuffles: `groupBy`, `join`, `distinct`, `repartition`, `orderBy`, and window functions with a
partition key. Narrow transformations (`filter`, `select`, `withColumn`, `union` of
same-partitioning data) do not.

**2. Narrow vs wide transformations - give examples and consequences.**

Narrow: each output partition depends on exactly one input partition (`map`, `filter`, `select`,
`withColumn`, `union`). No data movement; pipelined inside a stage. Wide: output partitions
depend on many input partitions (`groupBy`, `join`, `distinct`, `repartition`, `sort`). These
create a new stage, a shuffle, and a barrier. Rule of thumb: every `Exchange` node in
`explain()` marks a wide operation.

**3. When does a broadcast join help, and what are its limits?**

It helps when one side is small enough to fit in executor memory: Spark ships that side to every
executor and avoids shuffling the large side, turning a sort-merge join into a map-side broadcast
join. Limits: the broadcast side must fit in memory on **every** executor (default threshold
`spark.sql.autoBroadcastJoinThreshold=10MB`, and you can hint with `F.broadcast`); too large and
you get OOM or a silently ignored hint; broadcast also does not help if the *large* side is
skewed; and broadcasts are re-sent per stage/executor, so a "small" 500 MB table is not actually
small.

**4. What is Adaptive Query Execution and what problem does it solve?**

AQE re-optimises a query at runtime using actual shuffle statistics, instead of relying only on
plan-time estimates (which are often wrong). It coalesces many small post-shuffle partitions
(solving the 200-partition default problem), splits skewed partitions for skewed joins, and can
switch a sort-merge join to broadcast when a side turns out to be small. Enable with
`spark.sql.adaptive.enabled=true` (default since Spark 3.2) plus `coalescePartitions.enabled` and
`skewJoin.enabled`.

**5. How do you handle data skew, and what can AQE not fix?**

Detect it: one task takes far longer, one partition's spill/bytes dwarfs the others, `explain`
shows an unbalanced exchange. Fixes, in order: let AQE's skew join split the hot partitions;
salting the hot key (append a random suffix 0..N to the key on both sides, aggregate, then
re-aggregate without the suffix); broadcast the small side if skew comes from the join;
`repartition` on a better key; or fix the root cause (a null/placeholder key such as `"-1"`
holding most rows). AQE cannot fix a **skewed source read** - if one input file or one Kafka
partition holds most of the data, you must rebalance at ingestion.

**6. How do you choose partition columns, and what is over-partitioning?**

Partition on a low-to-medium cardinality column that is almost always in the `WHERE` clause
(date, region), that produces roughly 100 MB-1 GB per partition, and whose values are evenly
distributed. Over-partitioning is millions of tiny directories (e.g. partitioning by `user_id`
or by date+hour+minute), which inflates metadata, listing latency, and task count, and defeats
the point. Alternative for high-cardinality queries: leave the table unpartitioned/bucketed and
rely on Parquet/Delta/Iceberg statistics for file skipping.

**7. When should you cache, and when should you not?**

Cache when a DataFrame or an expensive intermediate is reused by 2-3+ actions, or in an
iterative algorithm. Do not cache when it is used once, when the source is already a small
in-memory table, when you are about to write the result anyway, or when it would push the job
into memory pressure (spilled cache is often slower than recomputing a cheap lineage). Caching
is lazy - the first action materialises it - and you should `unpersist()` when done.

**8. UDFs: why are they slow, and what are the alternatives?**

A Python UDF serialises the function and every row between the JVM and a Python worker, runs row
by row, prevents Catalyst from optimising the expression, and blocks whole-stage code
generation. Alternatives: built-in `pyspark.sql.functions` (always first choice); higher-order
SQL functions (`transform`, `filter`, `aggregate`) on arrays/maps; pandas UDFs (`@pandas_udf`)
which vectorise per batch via Arrow; or a Scala/Java UDF for JVM-native execution. Reserve plain
Python UDFs for cold paths.

**9. What is checkpointing, and how does it differ from caching?**

Checkpointing writes the DataFrame to a reliable store (HDFS/S3) and **truncates the lineage**,
so recovery does not replay an ever-growing chain of transformations. Caching keeps the lineage
and lives only in executor memory/disk for the session. Use checkpoints for very long lineage
chains, iterative algorithms, and streaming (where `checkpointLocation` stores offsets and
state). Checkpoint files must be cleaned up separately; they are not freed by `unpersist`.

**10. How does Structured Streaming achieve exactly-once?**

Exactly-once needs three things together: a **replayable source** with committed offsets (Kafka,
or a file source), a **durable checkpoint** recording the offsets and state for each micro-batch,
and an **idempotent/transactional sink** (Delta or Iceberg tables, or a `foreachBatch` upsert
keyed on the batch/version). The checkpoint lets a restart resume without reprocessing; the
transactional sink makes a partially written batch invisible or safely overwritten, so
at-least-once delivery becomes exactly-once state.

**11. Driver vs executor memory - what lives where, and what are the common OOMs?**

The driver runs the `SparkSession`, the DAG scheduler, the Catalyst planner, and collects
results. It owns memory for `collect()`, `toPandas()`, broadcast variables under the broadcast
threshold, and long-lived objects in notebook loops. Executors run tasks, hold shuffle data,
cached blocks, and broadcast joins. Common OOMs: `collect()`/`toPandas()` on a large result
(driver OOM); caching too much or broadcast joins that are not actually small (executor OOM);
`groupBy` on a skewed key with too few partitions (executor OOM/spill); and Python UDF workers
exhausting memory when each row builds large objects.

**12. How do you deal with small files?**

Prevent them at write time with sensible `repartition`/`coalesce` (one file per output
partition), and run scheduled compaction: Delta `OPTIMIZE`, Iceberg `rewrite_data_files`, Hudi
compaction, or a `coalesce`/`repartition` rewrite job. On the read side, bound
`spark.sql.files.maxPartitionBytes` and avoid one task per file. Small files hurt planning time,
object-store listing latency, and per-file metadata overhead, and they break predicate pushdown
efficiency because each file is opened individually.

**13. DataFrames vs RDDs - when would you still use an RDD?**

DataFrames go through Catalyst (rule-based + cost-based optimisation), Tungsten (code generation,
off-heap memory, compact binary layout), and a typed schema, so they are almost always faster and
safer. RDDs give you arbitrary Python/Scala objects and fine control over partitioning, which is
occasionally needed for custom iterative algorithms, untyped binary processing, or porting legacy
code. The interview answer: default to DataFrames, reach for RDDs (`df.rdd`) only when the API
genuinely cannot express the logic, and know that leaving the DataFrame API loses all optimizer
benefits.

**14. `repartition` vs `coalesce`?**

`repartition(n)` performs a **full shuffle** and produces `n` partitions of roughly even size;
use it to increase parallelism or to rebalance badly skewed data, optionally on a key
(`repartition(n, "key")`). `coalesce(n)` is narrow: it merges existing partitions without a
shuffle, so it can only decrease the count and the resulting partitions may be uneven; use it
before a write to reduce file count. Common pattern: `repartition` for balance, then
`coalesce`/`repartition` before writing to control file sizes.

**15. How do you debug a slow Spark job?**

Start from the Spark UI: find the longest stage, then the slowest task inside it, and compare its
input size and spill to its peers - a large gap means skew. Read `explain()` for `Exchange`
nodes, broadcast vs sort-merge joins, and full scans where you expected pruning. Check partition
counts (`spark.sql.shuffle.partitions`, input file counts), shuffle spill-to-disk metrics, GC
time, and whether a UDF or `collect()` is on the hot path. Then fix the highest-leverage thing:
broadcast/`repartition` the join, enable AQE, compact files, remove the UDF, or add a `filter`
earlier so less data is read.

## Running this notebook with Spark for real

```bash
# from the repo root
uv sync --extra notebooks --extra notebooks-spark
java -version                      # must be 17+
python3 scripts/run_notebooks.py notebooks/04_pyspark_concepts.ipynb
```

The runner executes every cell and saves the outputs back into the notebook. With PySpark and
Java present, `SPARK_AVAILABLE` becomes `True`, all SKIPPED messages turn into real results, and
the self-check assertion blocks run for real.

**Expected summary with Spark:** `Spark available: True`, `SparkSession stopped.`
**Expected summary here:** `Spark available: False`, all Spark cells reported SKIPPED.

In [80]:
print("Spark available:", SPARK_AVAILABLE)
if SPARK_AVAILABLE:
    print("All Spark cells executed against a local SparkSession.")
    print("Re-run this notebook whenever you change the code to refresh the saved outputs.")
else:
    print("All Spark cells were skipped; every concept above still includes its expected output.")
    if not HAS_PYSPARK:
        print("Missing: pyspark  ->  uv sync --extra notebooks --extra notebooks-spark")
    if not (HAS_JAVA and JAVA_RUNS):
        print("Missing: a working Java 17+ runtime  ->  install Temurin or openjdk@17")


Spark available: False
All Spark cells were skipped; every concept above still includes its expected output.
Missing: a working Java 17+ runtime  ->  install Temurin or openjdk@17


### Next steps

* `notebooks/03_sql_interview_questions.ipynb` - SQL window functions against a SQLite sandbox.
* `notebooks/05_interview_questions.ipynb` - a mixed question bank for mock interviews.

In [81]:
if SPARK_AVAILABLE:
    spark.stop()
    print("SparkSession stopped.")
else:
    print("SKIPPED - install PySpark + Java 17 and re-run this cell")


SKIPPED - install PySpark + Java 17 and re-run this cell
